In [ ]:
import pytrinamic
from pytrinamic.connections import ConnectionManager
from pytrinamic.modules import TMCM6110
import time
import scipy.io
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import trange
from itertools import chain

# Motor Control 

In [ ]:
class XYZ():
    def __init__(self):
        self.connectionManager = ConnectionManager()
        self.interface = self.connectionManager.connect()

        # Create an instance of the TMCM_6110 class
        self.module = TMCM6110(self.interface)


        self.motor_0 =  self.module.motors[0]
        self.motor_1 =  self.module.motors[1]
        self.motor_2 =  self.module.motors[2]
        print("Preparing parameters")
        
    def XYZ_setup(self,max_current=500,standby_current=200,boost_current=0,velocity=1500,acceleration=1000,position=0):
        motors_list = []
        for i in [0,1,2]:
            motor_name = f"motor_{i}"
            motor = getattr(self, motor_name)
            # Now you can use 'motor' as a reference to self.motor_0, self.motor_1, etc.
            motor.drive_settings.max_current=max_current
            motor.drive_settings.standby_current=standby_current
            motor.drive_settings.boost_current=boost_current
            motor.drive_settings.microstep_resolution = motor.ENUM.microstep_resolution_256_microsteps
            motor.max_acceleration=acceleration
            motor.max_velocity=velocity
#             motor.actual_position=position
            print(motor)
            motors_list.append(motor)
        return motors_list[2],motors_list[1],motors_list[0],self.interface
        
  

# Capturing functions


In [ ]:
def capture_location(scope, pt_exp, exp_name, x, y, num=1000):
    """
    Captures and stores traces for a given (x, y) location.

    Parameters:
    scope: Object responsible for capturing traces.
    pt_exp: Experiment dataset handler for plaintexts and keys.
    exp_name: Dataset handler where captured traces will be stored.
    x, y: Coordinates representing the capture location.
    num (int, optional): Number of traces to capture (default is 1000).
    """

    # Retrieve key, random plaintext, and fixed plaintext datasets
    keys_pt = pt_exp.get_dataset("keys").read_data(0, num)
    random_pt = pt_exp.get_dataset("plaintexts").read_data(0, num)
    fixed_pt = pt_exp.get_dataset("fixed_pt").read_data(0, num)

    # Capture traces using test vector leakage assessment (TVLA) method
    f, r = scope.capture_traces_tvla(num, keys_pt, fixed_pt, keys_pt, random_pt)

    # Store captured traces for the given location
    print("Storing for location: " + str(x) + "_" + str(y))
    exp_name.add_dataset("fixed_" + str(x) + "_" + str(y), f, datatype="float32")
    exp_name.add_dataset("random_" + str(x) + "_" + str(y), r, datatype="float32")

    print("Traces stored")

    return None


def Grid_Tracing_scapegoat(X_range, Y_range, X_number_of_step, Y_number_of_step, X, Y, Z, interface, scope, pt_exp, exp_store, number_of_traces):
    """
    Performs grid-based scanning and captures traces at each step.

    Parameters:
    X_range, Y_range: Step sizes for movement in X and Y directions.
    X_number_of_step, Y_number_of_step: Number of steps to take in X and Y directions.
    X, Y, Z: Actuators controlling movement along respective axes.
    interface: Communication interface for device control.
    scope: Object responsible for capturing traces.
    pt_exp: Experiment dataset handler for plaintexts and keys.
    exp_store: Experiment object to store captured traces.
    number_of_traces: Number of traces to capture at each grid point.
    """

    cordinate_traces = {}  # Dictionary to store traces at different coordinates
    X_moment = 0
    Y_moment = 0

    # Capture initial location traces
    capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)

    # Store initial positions
    X_start_position = X.get_actual_position()
    Y_start_position = Y.get_actual_position()
    print(f"Starting Position - ({X_start_position}, {Y_start_position})")

    # Perform scanning along the Y-axis
    while Y_moment <= Y_number_of_step:
        X_initial_position = X.get_actual_position()
        Y_initial_position = Y.get_actual_position()

        # Move in the positive X direction
        for _ in range(X_number_of_step):
            X.move_by(X_range)
            print(f'Moving X to {X_initial_position + X_range}')
            while X.get_actual_position() != X_initial_position + X_range:
                time.sleep(0.1)  # Wait until movement is complete
            X_moment += 1
            capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
            X_initial_position = X.get_actual_position()

        if Y_moment == Y_number_of_step:
            break  # Stop if the last Y step is reached

        # Move in the positive Y direction
        Y.move_by(Y_range)
        Y_moment += 1
        print(f'Moving Y to {Y_initial_position + Y_range}')
        while Y.get_actual_position() != Y_initial_position + Y_range:
            time.sleep(0.1)  # Wait until movement is complete
        capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
        Y_initial_position = Y.get_actual_position()

        # Move in the negative X direction
        for _ in range(X_number_of_step):
            X.move_by(-X_range)
            print(f'Moving X to {X_initial_position - X_range}')
            while X.get_actual_position() != X_initial_position - X_range:
                time.sleep(0.1)  # Wait until movement is complete
            X_moment -= 1
            capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
            X_initial_position = X.get_actual_position()

        if Y_moment == Y_number_of_step:
            break  # Stop if the last Y step is reached

        # Move in the positive Y direction again
        Y.move_by(Y_range)
        Y_moment += 1
        print(f'Moving Y to {Y_initial_position + Y_range}')
        while Y.get_actual_position() != Y_initial_position + Y_range:
            time.sleep(0.1)  # Wait until movement is complete
        capture_location(scope, pt_exp, exp_store, X_moment, Y_moment, number_of_traces)
        Y_initial_position = Y.get_actual_position()

    # Return to the starting position
    X.move_to(X_start_position)
    while X.get_actual_position() != X_start_position:
        time.sleep(0.1)  # Wait until movement is complete

    Y.move_to(Y_start_position)
    while Y.get_actual_position() != Y_start_position:
        time.sleep(0.1)  # Wait until movement is complete

    print(f"Final Position - ({X.get_actual_position()}, {Y.get_actual_position()})")

    return None


# Metrics (one-click)

In [1]:
def plot_CEMA_heatmap(test, pt_exp, num, target_byte=0, grid_size=5,rotation_n = 0,lower_b = 0,upper_b = 10000, model = 1,fntsz=18):
    """
    Compute and visualize Correlation Electromagnetic Analysis (CEMA) results as a heatmap.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for correlation analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    
    Returns:
    - CEMA_guesses_rotated: Rotated array of best key guesses from CEMA.
    - CEMA_values_rotated: Rotated array of maximum correlation values from CEMA.
    """

    # Retrieve keys and plaintext datasets
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Initialize arrays to store CEMA results
    CEMA_values = np.zeros((grid_size, grid_size))  # Stores maximum correlation values
    CEMA_guesses = np.zeros((grid_size, grid_size))  # Stores corresponding key guesses

    # Perform CEMA analysis across the grid
    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
            upper_b = len(traces[0])
            # Perform CEMA to obtain correlation values and best key guess
            
            best_guess, max_correlation = scapegoat_cpa_byte(traces[:,lower_b:upper_b], keys, plaintexts, target_byte)
            
                
            # Store results
            CEMA_values[i, j] = max_correlation
            CEMA_guesses[i, j] = best_guess

    # Rotate the heatmap for correct visualization
    CEMA_values_rotated = np.rot90(CEMA_values, k=rotation_n )  # Rotate by 90 degrees clockwise
    CEMA_guesses_rotated = np.rot90(CEMA_guesses, k=rotation_n )

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title
    plt.title("CEMA Heatmap",fontsize=fntsz+4)
    plt.xlabel("Grid Column (j)",fontsize=fntsz+2)
    plt.ylabel("Grid Row (i)",fontsize=fntsz+2)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)

    # Display the plot
    plt.show()

    return CEMA_guesses_rotated, CEMA_values_rotated

def plot_CEMA_heatmap_models(test, pt_exp, num, target_byte=0, grid_size=5,rotation_n = 0,lower_b = 0,upper_b = 10000, model = 1,fontsz = 22,r1=6,r2=7):
    """
    Compute and visualize Correlation Electromagnetic Analysis (CEMA) results as a heatmap.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for correlation analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    
    Returns:
    - CEMA_guesses_rotated: Rotated array of best key guesses from CEMA.
    - CEMA_values_rotated: Rotated array of maximum correlation values from CEMA.
    """

    # Retrieve keys and plaintext datasets
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Initialize arrays to store CEMA results
    CEMA_values = np.zeros((grid_size, grid_size))  # Stores maximum correlation values
    CEMA_guesses = np.zeros((grid_size, grid_size))  # Stores corresponding key guesses
    
    correct_key = keys[0][target_byte]
    if model==2:
        c = AES(keys[0])
    # Initialize a matrix to store max CPA values across different key guesses
#     maxcpa_matrix = np.zeros((int(num / div), 256))
    if model==4:
        dick = intermediates_round6(plaintexts,keys[0],rond = r1)
#         print(f"rond={r1}")
        idx = column_indices(r2)

    # Perform CEMA analysis across the grid
    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
            upper_b = len(traces[0])
            
            k = correct_key
            if model ==1:
                leakage = leakage_model_hamming_weight(num_traces=num, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
            elif model ==2:
                
                leakage = hd_cpa_last_round(plaintext=plaintexts,num_traces=num,c=c,target_byte=target_byte,r1=r1,r2=r2)
            # Compute Pearson correlation
            elif model ==3:
#                 c = AES(keys[0])
                leakage = hw_cpa_last_round(plaintext=plaintexts,num_traces=num,c=c,target_byte=target_byte,r1=r1,r2=r2)
            elif model ==4:
                leakage = np.zeros(num, dtype=np.float64)
                for killme in range(num):

                    if(target_byte==0):
                        leakage[killme] = hamming_weight(dick[f"{col_tar}6"][killme,idx]).sum()
                    else:
                        leakage[killme] = hamming_weight(dick[f"{col_tar}6"][killme,idx]^dick[f"{col_tar}7"][killme,idx]).sum()


#                       else  print(leakage[:10])
            # Compute Pearson correlation
            correlation = pearson_correlation(leakage, traces[:num])

            # Perform CEMA to obtain correlation values and best key guess
            
#             best_guess, max_correlation = scapegoat_cpa_byte(traces[:,lower_b:upper_b], keys, plaintexts, target_byte)
                
                
            # Store results
            CEMA_values[i, j] = np.nanmax(np.abs(correlation))
#             CEMA_guesses[i, j] = best_guess

    # Rotate the heatmap for correct visualization
    CEMA_values_rotated = np.rot90(CEMA_values, k=rotation_n )  # Rotate by 90 degrees clockwise
#     CEMA_guesses_rotated = np.rot90(CEMA_guesses, k=rotation_n )

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title
    plt.title(f"CPA Heatmap (byte-{target_byte})",fontsize=fontsz + 4)
    plt.xlabel("Grid Column (j)",fontsize=fontsz + 2)
    plt.ylabel("Grid Row (i)",fontsize=fontsz + 2)
    plt.yticks(fontsize=fontsz )
    plt.xticks(fontsize=fontsz )
    fig= plt.gcf()
    # Display the plot
    plt.show()

    return CEMA_values_rotated,fig

def plot_SNR_min_heatmap(test, pt_exp, num, target_byte=0, grid_size=5, SNR_type="BYTE",rotation_n = 0,MAX=True):
    """
    Compute and visualize Signal-to-Noise Ratio (SNR) results as a heatmap.

    SNR Types:
    - "BYTE": Uses all possible byte combinations.
    - "FULL": Uses all possible 16-byte key combinations.
    - "HW": Uses the Hamming weight of the byte.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for SNR analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    - SNR_type: Type of SNR analysis ("BYTE", "FULL", or "HW").

    Returns:
    - SNR_values_rotated: Rotated array of maximum SNR values.
    - SNR_dB: SNR values in decibels (10 * log10 of SNR_values_rotated).
    """

    # Retrieve keys and plaintext datasets
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Select the labeling method based on SNR type
    if SNR_type == "BYTE":
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL":
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1]  # Extract the relevant labels for SNR computation
    else:
        print("Incorrect SNR type specified.")
        return -1

    # Initialize array to store SNR values
    SNR_values = np.zeros((grid_size, grid_size))
    SNR_min_values = np.zeros((grid_size, grid_size))
    

    # Get unique label values
    labels_unique = np.unique(labels)

    # Perform SNR analysis across the grid
    for i in range(grid_size):
        for j in trange(grid_size):
#             sorted_labels = {k: [] for k in labels_unique}  # Dictionary to store traces per label
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)

#             # Organize traces according to their label
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

#             # Compute SNR for the given labels and traces
#             snr_result = signal_to_noise_ratio(sorted_labels)
            if(MAX==False):
                result = compute_min_traces_from_traces(traces, labels)
                SNR_values[i, j] = np.nanmax(np.abs(result.get("theta")))
            else:
                result = compute_min_traces_max_snr(traces, labels)#signal_to_noise_ratio(sorted_labels)
#                 SNR_values[i, j] = np.nanmax(np.abs(result))
            # Store the maximum absolute SNR value
            SNR_values[i, j] = np.nanmax(np.abs(result.get("theta")))
#             SNR_min_values[i, j] = np.nanmax(np.abs(result.get("N_min")))

    # Rotate the heatmap for correct visualization
    SNR_values_rotated = np.rot90(SNR_values, k=rotation_n )  # Rotate by 90 degrees clockwise
    SNR_min_values_rotated = np.rot90(SNR_min_values, k=rotation_n )  # Rotate by 90 degrees clockwise

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
    sns.heatmap(SNR_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title based on SNR type
    plt.title(f"SNR Heatmap ({SNR_type} mode)")
    plt.xlabel("Grid Column (j)")
    plt.ylabel("Grid Row (i)")

    # Display the plot
    plt.show()
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(SNR_min_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title based on SNR type
    plt.title(f"Min number of traces for SNR ({SNR_type} mode)",fontsize=20)
    plt.xlabel("Grid Column (j)",fontsize=16)
    plt.ylabel("Grid Row (i)",fontsize=16)
    plt.yticks(fontsize=14)
    plt.xticks(fontsize=14)

    # Display the plot
    plt.show()

    # Compute SNR in decibels
    SNR_dB = 10 * np.log10(SNR_values_rotated)

    return SNR_values_rotated, SNR_dB , SNR_min_values_rotated


def plot_SNR_heatmap(test, pt_exp, num, target_byte=0, grid_size=5, SNR_type="BYTE",rotation_n = 0,fntsz=20):
    """
    Compute and visualize Signal-to-Noise Ratio (SNR) results as a heatmap.

    SNR Types:
    - "BYTE": Uses all possible byte combinations.
    - "FULL": Uses all possible 16-byte key combinations.
    - "HW": Uses the Hamming weight of the byte.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for SNR analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    - SNR_type: Type of SNR analysis ("BYTE", "FULL", or "HW").

    Returns:
    - SNR_values_rotated: Rotated array of maximum SNR values.
    - SNR_dB: SNR values in decibels (10 * log10 of SNR_values_rotated).
    """

    # Retrieve keys and plaintext datasets
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Select the labeling method based on SNR type
    if SNR_type == "BYTE":
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL":
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1]  # Extract the relevant labels for SNR computation
    else:
        print("Incorrect SNR type specified.")
        return -1

    # Initialize array to store SNR values
    SNR_values = np.zeros((grid_size, grid_size))

    # Get unique label values
    labels_unique = np.unique(labels)

    # Perform SNR analysis across the grid
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {k: [] for k in labels_unique}  # Dictionary to store traces per label
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)

            # Organize traces according to their label
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # Compute SNR for the given labels and traces
            snr_result = signal_to_noise_ratio(sorted_labels)

            # Store the maximum absolute SNR value
            SNR_values[i, j] = np.nanmax(np.abs(snr_result))

    # Rotate the heatmap for correct visualization
    SNR_values_rotated = np.rot90(SNR_values, k=rotation_n )  # Rotate by 90 degrees clockwise

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
    sns.heatmap(SNR_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title based on SNR type
    plt.title(f"SNR Heatmap ({SNR_type} mode)",fontsize=fntsz+4)
    plt.xlabel("Grid Column (j)",fontsize=fntsz+2)
    plt.ylabel("Grid Row (i)",fontsize=fntsz+2)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)
    # Display the plot
    fig= plt.gcf()
    plt.show()

    # Compute SNR in decibels
    SNR_dB = 10 * np.log10(SNR_values_rotated)

    return SNR_values_rotated, SNR_dB,fig


def plot_MI_heatmap(test, pt_exp, num,rotation_n = 0, target_byte=0, grid_size=5, MI_type="BYTE",num_columns = 100, sigma_traces=1, sigma_label=1,alpha =1.01, normalize = False,multiplier=100,r1=6,r2=7,nystrom=False,m_l=500,fntsz=20):
    """
    Compute and visualize Signal-to-Noise Ratio (SNR) results as a heatmap.

    SNR Types:
    - "BYTE": Uses all possible byte combinations.
    - "FULL": Uses all possible 16-byte key combinations.
    - "HW": Uses the Hamming weight of the byte.

    Parameters:
    - test: An object that provides access to trace datasets.
    - pt_exp: Experiment dataset handler for keys and plaintexts.
    - num: Number of traces to process.
    - target_byte: Byte index for SNR analysis (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    - SNR_type: Type of SNR analysis ("BYTE", "FULL", or "HW").

    Returns:
    - SNR_values_rotated: Rotated array of maximum SNR values.
    - SNR_dB: SNR values in decibels (10 * log10 of SNR_values_rotated).
    """

    # Retrieve keys and plaintext datasets
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)
    c = AES(keys[0])
    print("AES model generated")
    # Select the labeling method based on SNR type
    if MI_type == "BYTE":
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif MI_type =="HD":
        
        labels = hd_cpa_last_round(plaintext=plaintexts,num_traces=num,c=c,target_byte=target_byte,r1=r1,r2=r2)
        print("labels created")
    elif MI_type =="HD_byte":
        
        labels = hd_byte_cpa_last_round(plaintext=plaintexts,num_traces=num,c=c,target_byte=target_byte,r1=r1,r2=r2)
        print("HD byte labels created")
#         elif SNR_type == "HW":
#         labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
#     elif SNR_type == "FULL":
#         labels = Sbox[plaintexts ^ keys]
#         labels = labels[:, 1]  # Extract the relevant labels for SNR computation
    else:
        print("Incorrect SNR type specified.")
        return -1

    # Initialize array to store SNR values
    SNR_values = np.zeros((grid_size, grid_size))

#     # Get unique label values
#     labels_unique = np.unique(labels)
#     all_cell_traces = None
#     for i in trange(grid_size):
#         for j in trange(grid_size):
#             traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
#             if all_cell_traces is None:
#                 all_cell_traces = traces
#             else:
#                 all_cell_traces = np.concatenate((all_cell_traces, traces), axis=0)

# #     all_traces = np.concatenate(all_cell_traces, axis=0)
    
#     global_med = np.median(all_cell_traces, axis=0)
#     global_mad = np.median(np.abs(all_cell_traces - global_med), axis=0)
#     print(all_cell_traces.shape)

#     global_mad[global_mad == 0] = 1.0    


    # Perform SNR analysis across the grid
    for i in trange(grid_size):
        for j in trange(grid_size):
#             sorted_labels = {k: [] for k in labels_unique}  # Dictionary to store traces per label
#             print(f"Processing grid position ({i}, {j})")
            
            # Retrieve traces for the current grid position
            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)

#             # Organize traces according to their label
#             for index, label in enumerate(labels):
#                 sorted_labels[label].append(np.array(traces[index]))
#             med = np.median(traces, axis=0)
#             mad = np.median(np.abs(traces - med), axis=0)
#             mad[mad == 0] = 1.0     # avoid divide-by-zero
#             traces = (traces - med) / mad
#             traces = np.nan_to_num(traces)
#             traces = np.nan_to_num((traces - traces.mean(0)) / (traces.std(0) + 1e-12))





# 
#             traces = (traces - global_med) / global_mad

            # Compute SNR for the given labels and traces
            snr_result = calculate_MI_EM_percell(traces[:1000 ,: ],labels[:1000],alpha,normalize,nystrom=nystrom,m_l=m_l)

            # Store the maximum absolute SNR value
            SNR_values[i, j] = snr_result

    # Rotate the heatmap for correct visualization
    SNR_values_rotated = np.rot90(SNR_values, k=rotation_n )  # Rotate by 90 degrees clockwise

    # Create a heatmap visualization
    plt.figure(figsize=(8, 6))
    sns.heatmap(SNR_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title based on SNR type
    plt.title(f"MI Heatmap ({MI_type} mode Byte - {target_byte})",fontsize=fntsz+4)
    plt.xlabel("Grid Column (j)",fontsize=fntsz+2)
    plt.ylabel("Grid Row (i)",fontsize=fntsz+2)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)
    # Display the plot
    fig= plt.gcf()
    plt.show()


    return SNR_values_rotated,snr_result,fig


def plot_t_statistic_heatmap(test, grid_size=5,rotation_n = 0,fntsz=20,title = False):
    """
    Compute and visualize t-statistics as a heatmap.

    This function calculates t-statistics for each position in a grid 
    and generates a heatmap representation.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).

    Returns:
    - t_values_rotated: Rotated array of maximum absolute t-statistics.
    """

    # Initialize the t-statistics array
    t_values = np.zeros((grid_size, grid_size))

    # Compute t-statistics for each grid position
    for i in range(grid_size):
        for j in range(grid_size):
            t_stat, t_max = test.calculate_t_test(f"fixed_{i}_{j}", f"random_{i}_{j}")
            t_values[i, j] = np.nanmax(np.abs(t_stat))  # Store the maximum absolute t-statistic

    # Rotate the heatmap for correct visualization
    t_values_rotated = np.rot90(t_values, k=rotation_n)  # Rotate by 90 degrees clockwise

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(t_values_rotated, annot=True, cbar=True, square=True)

    # Add labels and title
    if title:
        plt.title("Heatmap of t-statistics",fontsize=fntsz+4)
    plt.xlabel("Grid Column (j)",fontsize=fntsz+2)
    plt.ylabel("Grid Row (i)",fontsize=fntsz+2)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)
    
    
    # Display the plot
#     plt.savefig(f"final_data//{config}_t_heatmap.png",format="png",bbox_inches="tight")
    fig= plt.gcf()
    plt.show()

    return t_values_rotated,fig


def generate_box_plots(test, pt_exp, num_list, target_byte=0, grid_size=5):
    """
    Generate box plots for SNR values at different numbers of traces.

    This function computes SNR values for various trace counts and visualizes them
    using box plots to analyze variations in SNR across different datasets.

    Parameters:
    - test: An object that provides the `plot_SNR_heatmap_byte_bp` function.
    - pt_exp: Experiment data handler.
    - num_list: A list of trace counts to evaluate.
    - target_byte: The target byte used for SNR calculation.
    - grid_size: The grid size for SNR calculations (default is 5x5).

    Returns:
    - all_values: A list containing SNR value lists for each trace count.
    """

    all_values = []  # Stores separate lists of SNR values for each trace count
    labels = []  # Labels corresponding to each dataset

    for num in num_list:
        print(f"Processing num_traces = {num}...")

        # Compute the SNR values for the given number of traces
        SNR_values_rotated = plot_SNR_heatmap_byte_bp(test, pt_exp, num, target_byte, grid_size)

        # Validate and flatten SNR values
        if isinstance(SNR_values_rotated, list):
            all_values.append(list(chain.from_iterable(SNR_values_rotated)))  # Flatten and store values
        else:
            print(f"Invalid data format for num_traces={num}: {SNR_values_rotated}")
            continue

        labels.append(f"{num} traces")  # Create labels for the box plot

    # Generate box plot if valid data is available
    if all_values:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=all_values)

        # Configure plot labels and title
        plt.xticks(ticks=range(len(num_list)), labels=labels)
        plt.xlabel("Number of Traces")
        plt.ylabel("SNR Values")
        plt.title("Box Plot of SNR Values for Different Trace Counts")

        # Display the plot
        plt.show()
    else:
        print("No valid data to plot.")

    return all_values

    

def plot_CEMA_wr(test, pt_exp, num, target_byte=0, x=0, y=0, div=10):
    """
    Generate a CPA correlation plot comparing the correct key vs. wrong keys over increasing trace counts.

    This function performs Correlation Power Analysis (CPA) across different numbers of traces, 
    visualizing how the correct key and wrong keys' correlation evolve.

    Parameters:
    - test: An object that provides trace datasets.
    - pt_exp: Experiment data handler providing plaintext and key datasets.
    - num: Total number of traces to analyze.
    - target_byte: The target byte index in the key (default is 0).
    - x, y: Grid position for selecting the dataset.
    - div: Step size for processing traces in intervals.

    Returns:
    - maxcpa_matrix: A matrix storing the maximum CPA correlation values for all 256 key guesses.
    """

    # Load key and plaintext data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Load power traces for the selected grid position
    traces = test.get_dataset(f"random_{x}_{y}").read_data(0, num)

    # Initialize a matrix to store max CPA values across different key guesses
    maxcpa_matrix = np.zeros((int(num / div), 256))

    iterations = 1
    for i in trange(1, num):
        if i % div == 0:
            for k in range(256):
                # Compute leakage model using Hamming weight
                leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
                
                # Compute Pearson correlation
                correlation = pearson_correlation(leakage, traces[:i])

                # Store max correlation value for this key guess
                maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))
            
            iterations += 1

    # Debugging: Check matrix shape
    print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

    # Prepare x-axis values (trace count steps)
    xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))

    # Plotting
    plt.figure(figsize=(10, 6))

    if maxcpa_matrix.shape[0] > 2:
        # Plot the statistical threshold
        plt.plot(xp, (abs(4) / np.sqrt(xp * div)) * np.ones_like(xp), 
                 color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

        # Plot CPA correlations for all 256 key hypotheses
        for i in range(256):
            if i == 43:  # Assuming 43 is the correct key
                plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red",
                         alpha=0.9, linewidth=1.5, label="Correct key")
            else:
                plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey",
                         alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")

    # Configure plot labels and title
    plt.xlabel(f"No. of traces × {div}", fontsize=14)
    plt.ylabel("Max CPA Value", fontsize=14)
    plt.title(f"Correlation Power Analysis: ({x}, {y})", fontsize=18)
    plt.yticks(fontsize=12)
    plt.xticks(fontsize=12)
    plt.legend()

    # Display the plot
    plt.show()

    return maxcpa_matrix



def plot_CEMA_traces(test, pt_exp, num, target_byte=0, x=0, y=0, div=10,fntsz=20):
    """
    Generate a CPA correlation plot comparing the correct key vs. wrong keys over increasing trace counts.

    This function performs Correlation Power Analysis (CPA) across different numbers of traces, 
    visualizing how the correct key and wrong keys' correlation evolve.

    Parameters:
    - test: An object that provides trace datasets.
    - pt_exp: Experiment data handler providing plaintext and key datasets.
    - num: Total number of traces to analyze.
    - target_byte: The target byte index in the key (default is 0).
    - x, y: Grid position for selecting the dataset.
    - div: Step size for processing traces in intervals.

    Returns:
    - maxcpa_matrix: A matrix storing the maximum CPA correlation values for all 256 key guesses.
    """

    # Load key and plaintext data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Load power traces for the selected grid position
    traces = test.get_dataset(f"random_{x}_{y}").read_data(0, num)

    # Initialize a matrix to store max CPA values across different key guesses
    maxcpa_matrix = np.zeros((int(num / div), 256))

    iterations = 1
    for i in trange(1, num):
        if i % div == 0:
            for k in range(256):
                # Compute leakage model using Hamming weight
                leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
                
                # Compute Pearson correlation
                correlation = pearson_correlation(leakage, traces[:i])

                # Store max correlation value for this key guess
                maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))
            
            iterations += 1

    # Debugging: Check matrix shape
    print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

    # Prepare x-axis values (trace count steps)
    xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))

    # Plotting
    plt.figure(figsize=(10, 6))

    if maxcpa_matrix.shape[0] > 2:
        # Plot the statistical threshold
        plt.plot(xp, (abs(4) / np.sqrt(xp * div)) * np.ones_like(xp), 
                 color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

        # Plot CPA correlations for all 256 key hypotheses
        for i in range(256):
            if i == 43:  # Assuming 43 is the correct key
                plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red",
                         alpha=0.9, linewidth=1.5, label="Correct key")
            else:
                plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey",
                         alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")

    # Configure plot labels and title
    plt.xlabel(f"No. of traces × {div}", fontsize=fntsz+2)
    plt.ylabel("Max CPA Value", fontsize=fntsz+2)
    plt.title(f"Correlation Power Analysis: ({x}, {y})", fontsize=fntsz+4)
    plt.yticks(fontsize=fntsz)
    plt.xticks(fontsize=fntsz)
    plt.legend()

    # Display the plot
    plt.show()

    return maxcpa_matrix

def reverse_coords_ccw(x_rot, y_rot, shape, k):
    """
    Given a point (x_rot, y_rot) in an array that was created as
        rotated = np.rot90(original, k)
    (i.e., original was rotated counter-clockwise k times),
    return the corresponding (x_orig, y_orig) in the original array.

    Parameters
    ----------
    x_rot, y_rot : int
        Coordinates in the rotated image.
    shape : tuple
        Shape of the original array as (rows, cols).
    k : int
        The `k` passed to np.rot90 (number of 90° CCW rotations applied).

    Returns
    -------
    (x_orig, y_orig)
    """
    # np.rot90 with k CCW is equivalent to (4 - k) clockwise rotations.
    n_clockwise = (k) % 4  # same as (4 - k) % 4
    rows, cols = shape
    if n_clockwise == 0:
        return x_rot, y_rot
    elif n_clockwise == 1:  # 90° clockwise
        return y_rot, cols - 1 - x_rot
    elif n_clockwise == 2:  # 180°
        return rows - 1 - x_rot, cols - 1 - y_rot
    elif n_clockwise == 3:  # 90° counter-clockwise
        return rows - 1 - y_rot, x_rot


## helper functions  

In [1]:
import numpy as np
from scipy.stats import norm

def compute_min_traces_max_snr(traces: np.ndarray, labels: np.ndarray,
                                target_alpha: float = 1e-6, target_beta: float = 1e-3) -> dict:
    """
    Computes minimum number of traces using the maximum SNR across time samples.
    
    Args:
        traces (np.ndarray): Array of shape (N, T)
        labels (np.ndarray): Array of shape (N,) with byte values (0–255)
        target_alpha (float): Desired false positive rate
        target_beta (float): Desired false negative rate
        
    Returns:
        dict: Contains max_snr (theta), snr array, gamma, alpha, beta, N_min
    """

    N, T = traces.shape
    unique_classes = np.unique(labels)
    K = len(unique_classes)

    # Compute per-class means and variances per time sample
    class_means = np.zeros((K, T))
    class_vars = np.zeros((K, T))
    class_counts = np.zeros(K)

    for i, cls in enumerate(unique_classes):
        cls_traces = traces[labels == cls]
        class_counts[i] = len(cls_traces)
        class_means[i] = np.mean(cls_traces, axis=0)
        class_vars[i] = np.var(cls_traces, axis=0, ddof=1)

    # Global mean
    global_mean = np.average(class_means, axis=0, weights=class_counts)

    # Inter-class variance (signal) per time index
    signal = np.average((class_means - global_mean) ** 2, axis=0, weights=class_counts)

    # Intra-class variance (noise) per time index
    noise = np.average(class_vars, axis=0, weights=class_counts)

    # SNR per time index
    snr = signal / noise
    theta = np.max(snr)

    # Step 1: detection threshold under H0
    mu_z0 = K / N
    sigma_z0 = np.sqrt(2 * K / N**2)
    gamma = mu_z0 + sigma_z0 * norm.ppf(1 - target_alpha)

    # Step 2: false positive and false negative rates
    mu_z1 = theta + mu_z0
    sigma_z1 = np.sqrt(4 * theta / N + 2 * K / N**2)

    alpha_est = norm.sf((gamma - mu_z0) / sigma_z0)
    beta_est = 1 - norm.sf((gamma - mu_z1) / sigma_z1)

    # Step 3: closed-form N_min from Equation (18)
    a = norm.ppf(1 - target_alpha)
    b = norm.ppf(1 - target_beta)
    sqrt_2K = np.sqrt(2 * K)
    x0 = 2 * b**2 + a * sqrt_2K + abs(b) * np.sqrt(2 * K + 4 * b**2 + 4 * a * sqrt_2K)
    N_min =0 #int(np.ceil(x0 / theta))

    return {
        'theta_max': theta,
        'snr': snr,
        'gamma': gamma,
        'alpha': alpha_est,
        'beta': beta_est,
        'N_min': N_min
    }

# def compute_min_traces_max_snr(traces: np.ndarray, labels: np.ndarray,
#                                 target_alpha: float = 1e-6, target_beta: float = 1e-3) -> dict:
#     """
#     Computes minimum number of traces using the maximum SNR across time samples.
    
#     Args:
#         traces (np.ndarray): Array of shape (N, T)
#         labels (np.ndarray): Array of shape (N,) with byte values (0–255)
#         target_alpha (float): Desired false positive rate
#         target_beta (float): Desired false negative rate
        
#     Returns:
#         dict: Contains max_snr (theta), snr array, gamma, alpha, beta, N_min
#     """

#     N, T = traces.shape
#     unique_classes = np.unique(labels)
#     K = len(unique_classes)

#     # Compute per-class means and variances per time sample
#     class_means = np.zeros((K, T))
#     class_vars = np.zeros((K, T))
#     class_counts = np.zeros(K)

#     for i, cls in enumerate(unique_classes):
#         cls_traces = traces[labels == cls]
#         class_counts[i] = len(cls_traces)
#         class_means[i] = np.mean(cls_traces, axis=0)
#         class_vars[i] = np.var(cls_traces, axis=0, ddof=1)

#     # Global mean
#     global_mean = np.average(class_means, axis=0, weights=class_counts)

#     # Inter-class variance (signal) per time index
#     signal = np.average((class_means - global_mean) ** 2, axis=0, weights=class_counts)

#     # Intra-class variance (noise) per time index
#     noise = np.average(class_vars, axis=0, weights=class_counts)

#     # SNR per time index
#     snr = signal / noise
#     theta = np.max(snr)

#     # Step 1: detection threshold under H0
#     mu_z0 = K / N
#     sigma_z0 = np.sqrt(2 * K / N**2)
#     gamma = mu_z0 + sigma_z0 * norm.ppf(1 - target_alpha)

#     # Step 2: false positive and false negative rates
#     mu_z1 = theta + mu_z0
#     sigma_z1 = np.sqrt(4 * theta / N + 2 * K / N**2)

#     alpha_est = norm.sf((gamma - mu_z0) / sigma_z0)
#     beta_est = 1 - norm.sf((gamma - mu_z1) / sigma_z1)

#     # Step 3: closed-form N_min from Equation (18)
#     a = norm.ppf(1 - target_alpha)
#     b = norm.ppf(1 - target_beta)
#     sqrt_2K = np.sqrt(2 * K)
#     x0 = 2 * b**2 + a * sqrt_2K + abs(b) * np.sqrt(2 * K + 4 * b**2 + 4 * a * sqrt_2K)
    
#     if theta is None or np.isnan(theta) or theta <= 0:
#         N_min = np.inf  # or np.nan, depending on what makes more sense downstream
#     else:
#         N_min = int(np.ceil(x0 / theta))

#     return {
#         'theta_max': theta,
#         'snr': snr,
#         'gamma': gamma,
#         'alpha': alpha_est,
#         'beta': beta_est,
#         'N_min': N_min
#     }

def compute_min_traces_from_traces(traces: np.ndarray, labels: np.ndarray,
                                    target_alpha: float = 1e-6, target_beta: float = 1e-3) -> dict:
    """
    Compute minimum number of traces required for SNR-based leakage detection using the byte value model.
    
    Args:
        traces (np.ndarray): Array of shape (num_traces, trace_length)
        labels (np.ndarray): Array of shape (num_traces,) with byte values (0-255)
        target_alpha (float): Desired false positive rate
        target_beta (float): Desired false negative rate

    Returns:
        dict: Dictionary containing theta, gamma, alpha, beta, and N_min
    """

    # Get unique classes (e.g., 256 possible byte values)
    unique_classes = np.unique(labels)
    K = len(unique_classes)

    # Compute class means and counts
    class_means = []
    class_counts = []
    for cls in unique_classes:
        cls_traces = traces[labels == cls]
        class_counts.append(len(cls_traces))
        class_means.append(np.mean(cls_traces, axis=0))

    class_means = np.array(class_means)
    class_counts = np.array(class_counts)
    overall_mean = np.average(class_means, axis=0, weights=class_counts)

    # Signal variance (inter-class)
    signal_var = np.sum(class_counts * np.sum((class_means - overall_mean)**2, axis=1)) / np.sum(class_counts)

    # Noise variance (intra-class)
    pooled_noise = 0
    for cls, count in zip(unique_classes, class_counts):
        cls_traces = traces[labels == cls]
        cls_mean = np.mean(cls_traces, axis=0)
        pooled_noise += np.sum((cls_traces - cls_mean) ** 2)

    noise_var = pooled_noise / np.sum(class_counts)

    # Estimated SNR (theta)
    theta = signal_var / noise_var

    # Estimate current number of traces
    N_est = len(traces)

    # Step 1: Detection threshold gamma under H0
    mu_z0 = K / N_est
    sigma_z0 = np.sqrt(2 * K / N_est**2)
    gamma = mu_z0 + sigma_z0 * norm.ppf(1 - target_alpha)

    # Step 2: Evaluate actual alpha and beta using estimated gamma
    mu_z1 = theta + mu_z0
    sigma_z1 = np.sqrt(4 * theta / N_est + 2 * K / N_est**2)

    alpha_est = norm.sf((gamma - mu_z0) / sigma_z0)
    beta_est = 1 - norm.sf((gamma - mu_z1) / sigma_z1)

    # Step 3: Compute N_min using Equation 18
    a = norm.ppf(1 - target_alpha)
    b = norm.ppf(1 - target_beta)
    sqrt_2K = np.sqrt(2 * K)
    x0 = 2 * b**2 + a * sqrt_2K + abs(b) * np.sqrt(2 * K + 4 * b**2 + 4 * a * sqrt_2K)

    if theta is None or np.isnan(theta) or theta <= 0:
        N_min = np.inf  # or np.nan, depending on what makes more sense downstream
    else:
        N_min = int(np.ceil(x0 / theta))


    return {
        'theta': theta,
        'gamma': gamma,
        'alpha': alpha_est,
        'beta': beta_est,
        'N_min': N_min
    }

def scapegoat_cpa(experiment):
    """
    Perform Correlation Power Analysis (CPA) on an experiment dataset.

    Parameters:
    - experiment: An object containing datasets for traces, keys, and plaintexts.

    Returns:
    - best_guess: List of best key byte guesses (one per byte position).
    - cpa_refs: List of highest correlation values for each key byte position.
    """
    num_bytes = 16  # AES has 16 key bytes
    max_cpa = np.zeros(256)  # Store CPA values for each subkey guess
    cpa_refs = np.zeros(num_bytes)  # Highest CPA values per key byte
    best_guess = np.zeros(num_bytes, dtype=int)  # Best key guesses

    # Load datasets
    traces = experiment.get_dataset("CW_Capture_Traces").read_all()
    keys = experiment.get_dataset("CW_Capture_Keys").read_all()
    plaintexts = experiment.get_dataset("CW_Capture_Plaintexts").read_all()

    # Perform CPA attack for each byte in the key
    for byte_idx in trange(num_bytes, desc="CPA on key bytes"):
        for k in range(256):
            # Compute leakage model
            leakage = leakage_model_hamming_weight(
                num_traces=len(plaintexts),
                plaintexts=plaintexts,
                subkey_guess=k,
                target_byte=byte_idx
            )
            # Compute Pearson correlation
            correlation = pearson_correlation(leakage, traces)
            max_cpa[k] = np.nanmax(np.abs(correlation))

        # Store best guess and highest CPA value for this byte position
        best_guess[byte_idx] = np.argmax(max_cpa)
        cpa_refs[byte_idx] = np.nanmax(max_cpa)

    return best_guess, cpa_refs

def scapegoat_cpa_byte(traces, keys, plaintexts, target_byte):
    """
    Perform Correlation Power Analysis (CPA) on a specific key byte to guess the subkey.

    Parameters:
    - traces: The captured power traces.
    - keys: The actual secret keys corresponding to the traces.
    - plaintexts: The plaintexts used for the power analysis.
    - target_byte: The index of the target byte in the key to analyze.

    Returns:
    - best_guess: The best guess for the target key byte.
    - cpa_ref: The highest correlation value obtained for the target byte.
    """
    max_cpa = np.zeros(256)  # Store maximum CPA values for each possible subkey guess
    cpa_ref = 0  # Store highest correlation value for the target byte
    best_guess = 0  # Store best subkey guess for the target byte

    # Perform CPA attack for each possible subkey guess (0-255)
    for k in range(256):
        # Compute leakage model for each subkey guess
        leakage = leakage_model_hamming_weight(
            num_traces=len(plaintexts),
            plaintexts=plaintexts,
            subkey_guess=k,
            target_byte=target_byte
        )
        # Compute the Pearson correlation between the leakage and the traces
        correlation = pearson_correlation(leakage, traces)
        max_cpa[k] = np.nanmax(np.abs(correlation))  # Store the highest correlation for this guess

    # Find the best subkey guess and highest correlation value
    best_guess = np.argmax(max_cpa)
    cpa_ref = np.nanmax(max_cpa)

    return best_guess, cpa_ref

def leakage_model_hamming_weight_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = bin(Sbox[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]).count('1')

    return leakage

def sbox_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = Sbox[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]

    return leakage

def no_sbox_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = S[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]

    return leakage

def plot_heatmap(heatmap_values, text, anno=False, cba=True, squar=True):
    """
    Plots a heatmap using seaborn.

    Parameters:
    - heatmap_values: A 2D array or matrix of values to display in the heatmap.
    - text: Title for the heatmap.
    - anno: Boolean flag to display annotations on the heatmap (default is False).
    - cba: Boolean flag to display the color bar (default is True).
    - squar: Boolean flag to make the plot square-shaped (default is True).
    
    Returns:
    - None
    """
    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_values, annot=anno, cbar=cba, square=squar)

    # Adding labels and title
    plt.title(text)
    plt.xlabel("X")
    plt.ylabel("Y")

    # Show the plot
    plt.show()
    return None

    
def save_mat(matr, file_name):
    """
    Saves a given matrix to a .mat file.

    Parameters:
    - matr: The matrix to be saved (should be a numpy array).
    - file_name: The name of the file to save the matrix to (including the .mat extension).

    Returns:
    - None
    """
    # Save the matrix to a .mat file using scipy's savemat function
    scipy.io.savemat(file_name, {'matrix': matr})    
    
    return None


def save_fig(filename):
    """
    Saves the current figure to a specified file in SVG format.

    Parameters:
    - filename: The name of the file (should include the file extension, e.g., '.svg').

    Returns:
    - None
    """
    # Save the current figure to the specified file with SVG format
    plt.gcf().savefig(filename, format="svg", bbox_inches="tight")
    
    return None



def plot_SNR_heatmap_byte_bp(test, pt_exp, num, target_byte=0, grid_size=5, SNR_type="BYTE"):
    """
    Calculate the Signal-to-Noise Ratio (SNR) for a grid and plot the results.

    Parameters:
    - test: An object that handles data retrieval.
    - pt_exp: Experiment data handler for the plaintexts and keys.
    - num: The number of traces to consider.
    - target_byte: The byte to target in the S-box (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    - SNR_type: The type of SNR to compute ("BYTE", "HW", or "FULL").
    
    Returns:
    - CEMA_values: The calculated SNR values for each grid location.
    """
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)
    
    # Initialize the list to store SNR values
    CEMA_values = []

    # Select the appropriate SNR calculation based on the SNR_type
    if SNR_type == "BYTE":
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL":
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1]
    else:
        print("Incorrect SNR type")
        return -1

    # Get unique labels for the SNR calculation
    labelsUnique = np.unique(labels)

    # Calculate SNR for each grid location
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {k: [] for k in labelsUnique}  # Initialize sorted labels dictionary
            print(f"loop_{i}_{j}")

            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
            
            # Organize the traces based on the labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # Calculate the SNR for the current grid point
            snr_value = signal_to_noise_ratio(sorted_labels)

            # Store the SNR value
            CEMA_values.append(snr_value)

    return CEMA_values

def test_to_avg(test,test_avg,avg=10,grid_size =11):

    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            rand = test.get_dataset(f"random_{i}_{j}").read_all()
            fix = test.get_dataset(f"fixed_{i}_{j}").read_all()
            
            # Reshape so each group of 10 rows becomes one block
            f = fix.reshape(-1, avg, fix.shape[1]).mean(axis=1)
            r = rand.reshape(-1, avg, rand.shape[1]).mean(axis=1)
            
            test_avg.add_dataset("fixed_" + str(i) + "_" + str(j), f, datatype="float32")
            test_avg.add_dataset("random_" + str(i) + "_" + str(j), r, datatype="float32")
            
def test_to_downsample(test, test_avg, step=10, grid_size=11):

    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"Processing grid position ({i}, {j})")

            # Retrieve traces for the current grid position
            rand = test.get_dataset(f"random_{i}_{j}").read_all()
            fix = test.get_dataset(f"fixed_{i}_{j}").read_all()
            
            # Select every 10th row instead of averaging
            f = fix[::step]
            r = rand[::step]
            
            test_avg.add_dataset(f"fixed_{i}_{j}", f, datatype="float32")
            test_avg.add_dataset(f"random_{i}_{j}", r, datatype="float32")

def plot_CEMA_traces_temp(traces, pt_exp,num, target_byte=0, div=10, visualize_correct = True,model =1, r1 = 9, r2 = 8,col_tar = 'sbb_o6',title= False):
    """
    Generate a CPA correlation plot comparing the correct key vs. wrong keys over increasing trace counts.

    This function performs Correlation Power Analysis (CPA) across different numbers of traces, 
    visualizing how the correct key and wrong keys' correlation evolve.

    Parameters:
    - test: An object that provides trace datasets.
    - pt_exp: Experiment data handler providing plaintext and key datasets.
    - num: Total number of traces to analyze.
    - target_byte: The target byte index in the key (default is 0).
    - x, y: Grid position for selecting the dataset.
    - div: Step size for processing traces in intervals.

    Returns:
    - maxcpa_matrix: A matrix storing the maximum CPA correlation values for all 256 key guesses.
    """

#     # Load key and plaintext data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

#     # Load power traces for the selected grid position
#     traces = test.get_dataset(f"random_{x}_{y}").read_data(0, num)
    correct_key = keys[0][target_byte]
    # Initialize a matrix to store max CPA values across different key guesses
    maxcpa_matrix = np.zeros((int(num / div), 256))
    if model==4:
        dick = intermediates_round6(plaintexts,keys[0],rond = r1)
#         print(f"rond={r1}")
        idx = column_indices(r2)
#         print(f"{dick[col_tar][0:10,idx]}")

    iterations = 1
    for i in trange(1, num):
        if i % div == 0:
            if visualize_correct:
                k = correct_key
                if model ==1:
                    leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
                elif model ==2:
                    c = AES(keys[0])

                    leakage = hd_cpa_last_round(plaintext=plaintexts,num_traces=i,c=c,target_byte=target_byte,r1=r1,r2=r2)
                # Compute Pearson correlation
                elif model ==3:
                    c = AES(keys[0])
                    leakage = hw_cpa_last_round(plaintext=plaintexts,num_traces=i,c=c,target_byte=target_byte,r1=r1,r2=r2)
                elif model ==4:
                    
                    leakage = np.zeros(i, dtype=np.float64)
                    for killme in range(i):
                        
                        if(target_byte==0):
                            leakage[killme] = hamming_weight(dick[f"{col_tar}6"][killme,idx]).sum()
                        else:
                            leakage[killme] = hamming_weight(dick[f"{col_tar}6"][killme,idx]^dick[f"{col_tar}7"][killme,idx]).sum()

                                    
#                       else  print(leakage[:10])
                # Compute Pearson correlation
                correlation = pearson_correlation(leakage, traces[:i])

                # Store max correlation value for this key guess
                maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))
            else:
                for k in range(256):
                    # Compute leakage model using Hamming weight
                    leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)

                    # Compute Pearson correlation
                    correlation = pearson_correlation(leakage, traces[:i])

                    # Store max correlation value for this key guess
                    maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))

                iterations += 1

    # Debugging: Check matrix shape
    print(target_byte)
    print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

    if visualize_correct:
        xp =  np.arange(2,  maxcpa_matrix.shape[0])
    else:
        xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))

    # Plotting
    plt.figure(figsize=(10, 6))

    if maxcpa_matrix.shape[0] > 2:
        # Plot the statistical threshold
        plt.plot(xp, (abs(4) / np.sqrt(xp * div)) * np.ones_like(xp), color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

        # Plot CPA correlations for all 256 key hypotheses
        if visualize_correct:
            plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, int(correct_key)], color="red",    alpha=0.9, linewidth=1.5, label="Correct key")
        else:
            for i in range(256):
                if i == correct_key:  # Assuming 43 is the correct key
                    plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red",
                             alpha=0.9, linewidth=1.5, label="Correct key")
                else:
                    plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey",
                             alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")

    # Configure plot labels and title
    plt.xlabel(f"No. of traces × {div}", fontsize=20)
    plt.ylabel("Max CPA Value", fontsize=20)
    if title:
        plt.title(f"Correlation Power Analysis", fontsize=24)
    plt.yticks(fontsize=18)
    plt.xticks(fontsize=18)
    plt.legend()
    fig = plt.gcf()
    # Display the plot
    plt.show()

    if visualize_correct:
        plt.figure(figsize=(10, 6))
        plt.plot(correlation)
        plt.xlabel(f"Time Samples", fontsize=20)
        plt.ylabel("CPA Value", fontsize=20)
        if title:
            plt.title(f"Correlation Power Analysis - (byte {target_byte})", fontsize=24)
        plt.yticks(fontsize=18)
        plt.xticks(fontsize=18)
        plt.ylim(-0.3, 0.3) 
        plt.legend()

        # Display the plot
        plt.show()
    # --- Compute threshold crossing index ---
    cpa_curve = maxcpa_matrix[2:len(xp) + 2, int(correct_key)]
    threshold_curve = (abs(4) / np.sqrt(xp * div))

    # Find the first index where CPA exceeds threshold AND stays above afterwards
    cross_idx = None
    for idx in range(len(cpa_curve)):
        if cpa_curve[idx] >= threshold_curve[idx]:
            # Check if it stays above for all later points
            if np.all(cpa_curve[idx:] >= threshold_curve[idx:]):
                cross_idx = idx
                break

    if cross_idx is not None:
        min_traces_to_cross = xp[cross_idx] * div
        print(f"✅ CPA crosses threshold permanently at index {cross_idx} (≈ {min_traces_to_cross} traces)")
    else:
        min_traces_to_cross = None
        print("⚠️ CPA never stays permanently above the threshold.")

    return maxcpa_matrix, correlation,fig,min_traces_to_cross



In [1]:
import numpy as np
from scipy.stats import norm
# def compute_min_traces_max_snr(traces: np.ndarray, labels: np.ndarray,
#                                 target_alpha: float = 1e-6, target_beta: float = 1e-3) -> dict:
#     """
#     Computes minimum number of traces using the maximum SNR across time samples.
    
#     Args:
#         traces (np.ndarray): Array of shape (N, T)
#         labels (np.ndarray): Array of shape (N,) with byte values (0–255)
#         target_alpha (float): Desired false positive rate
#         target_beta (float): Desired false negative rate
        
#     Returns:
#         dict: Contains max_snr (theta), snr array, gamma, alpha, beta, N_min
#     """

#     N, T = traces.shape
#     unique_classes = np.unique(labels)
#     K = len(unique_classes)

#     # Compute per-class means and variances per time sample
#     class_means = np.zeros((K, T))
#     class_vars = np.zeros((K, T))
#     class_counts = np.zeros(K)

#     for i, cls in enumerate(unique_classes):
#         cls_traces = traces[labels == cls]
#         class_counts[i] = len(cls_traces)
#         class_means[i] = np.mean(cls_traces, axis=0)
#         class_vars[i] = np.var(cls_traces, axis=0, ddof=1)

#     # Global mean
#     global_mean = np.average(class_means, axis=0, weights=class_counts)

#     # Inter-class variance (signal) per time index
#     signal = np.average((class_means - global_mean) ** 2, axis=0, weights=class_counts)

#     # Intra-class variance (noise) per time index
#     noise = np.average(class_vars, axis=0, weights=class_counts)

#     # SNR per time index
#     snr = signal / noise
#     theta = np.max(snr)

#     # Step 1: detection threshold under H0
#     mu_z0 = K / N
#     sigma_z0 = np.sqrt(2 * K / N**2)
#     gamma = mu_z0 + sigma_z0 * norm.ppf(1 - target_alpha)

#     # Step 2: false positive and false negative rates
#     mu_z1 = theta + mu_z0
#     sigma_z1 = np.sqrt(4 * theta / N + 2 * K / N**2)

#     alpha_est = norm.sf((gamma - mu_z0) / sigma_z0)
#     beta_est = 1 - norm.sf((gamma - mu_z1) / sigma_z1)

#     # Step 3: closed-form N_min from Equation (18)
#     a = norm.ppf(1 - target_alpha)
#     b = norm.ppf(1 - target_beta)
#     sqrt_2K = np.sqrt(2 * K)
#     x0 = 2 * b**2 + a * sqrt_2K + abs(b) * np.sqrt(2 * K + 4 * b**2 + 4 * a * sqrt_2K)
#     if theta is None or np.isnan(theta) or theta <= 0:
#         N_min = np.inf  # or np.nan, depending on what makes more sense downstream
#     else:
#         N_min = int(np.ceil(x0 / theta))

#     return {
#         'theta_max': theta,
#         'snr': snr,
#         'gamma': gamma,
#         'alpha': alpha_est,
#         'beta': beta_est,
#         'N_min': N_min
#     }

def compute_min_traces_from_traces(traces: np.ndarray, labels: np.ndarray,
                                    target_alpha: float = 1e-6, target_beta: float = 1e-3) -> dict:
    """
    Compute minimum number of traces required for SNR-based leakage detection using the byte value model.
    
    Args:
        traces (np.ndarray): Array of shape (num_traces, trace_length)
        labels (np.ndarray): Array of shape (num_traces,) with byte values (0-255)
        target_alpha (float): Desired false positive rate
        target_beta (float): Desired false negative rate

    Returns:
        dict: Dictionary containing theta, gamma, alpha, beta, and N_min
    """

    # Get unique classes (e.g., 256 possible byte values)
    unique_classes = np.unique(labels)
    K = len(unique_classes)

    # Compute class means and counts
    class_means = []
    class_counts = []
    for cls in unique_classes:
        cls_traces = traces[labels == cls]
        class_counts.append(len(cls_traces))
        class_means.append(np.mean(cls_traces, axis=0))

    class_means = np.array(class_means)
    class_counts = np.array(class_counts)
    overall_mean = np.average(class_means, axis=0, weights=class_counts)

    # Signal variance (inter-class)
    signal_var = np.sum(class_counts * np.sum((class_means - overall_mean)**2, axis=1)) / np.sum(class_counts)

    # Noise variance (intra-class)
    pooled_noise = 0
    for cls, count in zip(unique_classes, class_counts):
        cls_traces = traces[labels == cls]
        cls_mean = np.mean(cls_traces, axis=0)
        pooled_noise += np.sum((cls_traces - cls_mean) ** 2)

    noise_var = pooled_noise / np.sum(class_counts)

    # Estimated SNR (theta)
    theta = signal_var / noise_var

    # Estimate current number of traces
    N_est = len(traces)

    # Step 1: Detection threshold gamma under H0
    mu_z0 = K / N_est
    sigma_z0 = np.sqrt(2 * K / N_est**2)
    gamma = mu_z0 + sigma_z0 * norm.ppf(1 - target_alpha)

    # Step 2: Evaluate actual alpha and beta using estimated gamma
    mu_z1 = theta + mu_z0
    sigma_z1 = np.sqrt(4 * theta / N_est + 2 * K / N_est**2)

    alpha_est = norm.sf((gamma - mu_z0) / sigma_z0)
    beta_est = 1 - norm.sf((gamma - mu_z1) / sigma_z1)

    # Step 3: Compute N_min using Equation 18
    a = norm.ppf(1 - target_alpha)
    b = norm.ppf(1 - target_beta)
    sqrt_2K = np.sqrt(2 * K)
    x0 = 2 * b**2 + a * sqrt_2K + abs(b) * np.sqrt(2 * K + 4 * b**2 + 4 * a * sqrt_2K)

    if theta is None or np.isnan(theta) or theta <= 0:
        N_min = np.inf  # or np.nan, depending on what makes more sense downstream
    else:
        N_min = int(np.ceil(x0 / theta))


    return {
        'theta': theta,
        'gamma': gamma,
        'alpha': alpha_est,
        'beta': beta_est,
        'N_min': N_min
    }

def scapegoat_cpa(experiment):
    """
    Perform Correlation Power Analysis (CPA) on an experiment dataset.

    Parameters:
    - experiment: An object containing datasets for traces, keys, and plaintexts.

    Returns:
    - best_guess: List of best key byte guesses (one per byte position).
    - cpa_refs: List of highest correlation values for each key byte position.
    """
    num_bytes = 16  # AES has 16 key bytes
    max_cpa = np.zeros(256)  # Store CPA values for each subkey guess
    cpa_refs = np.zeros(num_bytes)  # Highest CPA values per key byte
    best_guess = np.zeros(num_bytes, dtype=int)  # Best key guesses

    # Load datasets
    traces = experiment.get_dataset("CW_Capture_Traces").read_all()
    keys = experiment.get_dataset("CW_Capture_Keys").read_all()
    plaintexts = experiment.get_dataset("CW_Capture_Plaintexts").read_all()

    # Perform CPA attack for each byte in the key
    for byte_idx in trange(num_bytes, desc="CPA on key bytes"):
        for k in range(256):
            # Compute leakage model
            leakage = leakage_model_hamming_weight(
                num_traces=len(plaintexts),
                plaintexts=plaintexts,
                subkey_guess=k,
                target_byte=byte_idx
            )
            # Compute Pearson correlation
            correlation = pearson_correlation(leakage, traces)
            max_cpa[k] = np.nanmax(np.abs(correlation))

        # Store best guess and highest CPA value for this byte position
        best_guess[byte_idx] = np.argmax(max_cpa)
        cpa_refs[byte_idx] = np.nanmax(max_cpa)

    return best_guess, cpa_refs

def scapegoat_cpa_byte(traces, keys, plaintexts, target_byte):
    """
    Perform Correlation Power Analysis (CPA) on a specific key byte to guess the subkey.

    Parameters:
    - traces: The captured power traces.
    - keys: The actual secret keys corresponding to the traces.
    - plaintexts: The plaintexts used for the power analysis.
    - target_byte: The index of the target byte in the key to analyze.

    Returns:
    - best_guess: The best guess for the target key byte.
    - cpa_ref: The highest correlation value obtained for the target byte.
    """
    max_cpa = np.zeros(256)  # Store maximum CPA values for each possible subkey guess
    cpa_ref = 0  # Store highest correlation value for the target byte
    best_guess = 0  # Store best subkey guess for the target byte

    # Perform CPA attack for each possible subkey guess (0-255)
    for k in range(256):
        # Compute leakage model for each subkey guess
        leakage = leakage_model_hamming_weight(
            num_traces=len(plaintexts),
            plaintexts=plaintexts,
            subkey_guess=k,
            target_byte=target_byte
        )
        # Compute the Pearson correlation between the leakage and the traces
        correlation = pearson_correlation(leakage, traces)
        max_cpa[k] = np.nanmax(np.abs(correlation))  # Store the highest correlation for this guess

    # Find the best subkey guess and highest correlation value
    best_guess = np.argmax(max_cpa)
    cpa_ref = np.nanmax(max_cpa)

    return best_guess, cpa_ref

def leakage_model_hamming_weight_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = bin(Sbox[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]).count('1')

    return leakage

def sbox_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = Sbox[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]

    return leakage

def no_sbox_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = S[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]

    return leakage

def plot_heatmap(heatmap_values, text, anno=False, cba=True, squar=True):
    """
    Plots a heatmap using seaborn.

    Parameters:
    - heatmap_values: A 2D array or matrix of values to display in the heatmap.
    - text: Title for the heatmap.
    - anno: Boolean flag to display annotations on the heatmap (default is False).
    - cba: Boolean flag to display the color bar (default is True).
    - squar: Boolean flag to make the plot square-shaped (default is True).
    
    Returns:
    - None
    """
    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_values, annot=anno, cbar=cba, square=squar)

    # Adding labels and title
    plt.title(text)
    plt.xlabel("X")
    plt.ylabel("Y")

    # Show the plot
    plt.show()
    return None

    
def save_mat(matr, file_name):
    """
    Saves a given matrix to a .mat file.

    Parameters:
    - matr: The matrix to be saved (should be a numpy array).
    - file_name: The name of the file to save the matrix to (including the .mat extension).

    Returns:
    - None
    """
    # Save the matrix to a .mat file using scipy's savemat function
    scipy.io.savemat(file_name, {'matrix': matr})    
    
    return None


def save_fig(filename):
    """
    Saves the current figure to a specified file in SVG format.

    Parameters:
    - filename: The name of the file (should include the file extension, e.g., '.svg').

    Returns:
    - None
    """
    # Save the current figure to the specified file with SVG format
    plt.gcf().savefig(filename, format="svg", bbox_inches="tight")
    
    return None



def plot_SNR_heatmap_byte_bp(test, pt_exp, num, target_byte=0, grid_size=5, SNR_type="BYTE"):
    """
    Calculate the Signal-to-Noise Ratio (SNR) for a grid and plot the results.

    Parameters:
    - test: An object that handles data retrieval.
    - pt_exp: Experiment data handler for the plaintexts and keys.
    - num: The number of traces to consider.
    - target_byte: The byte to target in the S-box (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    - SNR_type: The type of SNR to compute ("BYTE", "HW", or "FULL").
    
    Returns:
    - CEMA_values: The calculated SNR values for each grid location.
    """
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)
    
    # Initialize the list to store SNR values
    CEMA_values = []

    # Select the appropriate SNR calculation based on the SNR_type
    if SNR_type == "BYTE":
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL":
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1]
    else:
        print("Incorrect SNR type")
        return -1

    # Get unique labels for the SNR calculation
    labelsUnique = np.unique(labels)

    # Calculate SNR for each grid location
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {k: [] for k in labelsUnique}  # Initialize sorted labels dictionary
            print(f"loop_{i}_{j}")

            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)
            
            # Organize the traces based on the labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # Calculate the SNR for the current grid point
            snr_value = signal_to_noise_ratio(sorted_labels)

            # Store the SNR value
            CEMA_values.append(snr_value)

    return CEMA_values
# def plot_CEMA_traces_temp(traces, keys, plaintexts,num, target_byte=0, div=10, visualize_correct = True):
#     """
#     Generate a CPA correlation plot comparing the correct key vs. wrong keys over increasing trace counts.

#     This function performs Correlation Power Analysis (CPA) across different numbers of traces, 
#     visualizing how the correct key and wrong keys' correlation evolve.

#     Parameters:
#     - test: An object that provides trace datasets.
#     - pt_exp: Experiment data handler providing plaintext and key datasets.
#     - num: Total number of traces to analyze.
#     - target_byte: The target byte index in the key (default is 0).
#     - x, y: Grid position for selecting the dataset.
#     - div: Step size for processing traces in intervals.

#     Returns:
#     - maxcpa_matrix: A matrix storing the maximum CPA correlation values for all 256 key guesses.
#     """

# #     # Load key and plaintext data
# #     keys = pt_exp.get_dataset("keys").read_data(0, num)
# #     plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

# #     # Load power traces for the selected grid position
# #     traces = test.get_dataset(f"random_{x}_{y}").read_data(0, num)
#     correct_key = keys[0][target_byte]
#     # Initialize a matrix to store max CPA values across different key guesses
#     maxcpa_matrix = np.zeros((int(num / div), 256))

#     iterations = 1
#     for i in trange(1, num):
#         if i % div == 0:
#             if visualize_correct:
#                 k = correct_key
#                 leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)

#                 # Compute Pearson correlation
#                 correlation = pearson_correlation(leakage, traces[:i])

#                 # Store max correlation value for this key guess
#                 maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))
#             else:
#                 for k in range(256):
#                     # Compute leakage model using Hamming weight
#                     leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)

#                     # Compute Pearson correlation
#                     correlation = pearson_correlation(leakage, traces[:i])

#                     # Store max correlation value for this key guess
#                     maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))

#                 iterations += 1

#     # Debugging: Check matrix shape
#     print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

#     if visualize_correct:
#         xp =  np.arange(2,  maxcpa_matrix.shape[0])
#     else:
#         xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))

#     # Plotting
#     plt.figure(figsize=(10, 6))

#     if maxcpa_matrix.shape[0] > 2:
#         # Plot the statistical threshold
#         plt.plot(xp, (abs(4) / np.sqrt(xp * div)) * np.ones_like(xp), 
#                  color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

#         # Plot CPA correlations for all 256 key hypotheses
#         if visualize_correct:
#             plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, int(correct_key)], color="red",
#                      alpha=0.9, linewidth=1.5, label="Correct key")
#         else:
#             for i in range(256):
#                 if i == correct_key:  # Assuming 43 is the correct key
#                     plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red",
#                              alpha=0.9, linewidth=1.5, label="Correct key")
#                 else:
#                     plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey",
#                              alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")

#     # Configure plot labels and title
#     plt.xlabel(f"No. of traces × {div}", fontsize=14)
#     plt.ylabel("Max CPA Value", fontsize=14)
#     plt.title(f"Correlation Power Analysis", fontsize=18)
#     plt.yticks(fontsize=12)
#     plt.xticks(fontsize=12)
#     plt.legend()

#     # Display the plot
#     plt.show()

#     if visualize_correct:
#         plt.figure(figsize=(10, 6))
#         plt.plot(correlation)
#         plt.xlabel(f"No. of traces × {div}", fontsize=14)
#         plt.ylabel("Max CPA Value", fontsize=14)
#         plt.title(f"Correlation Power Analysis", fontsize=18)
#         plt.yticks(fontsize=12)
#         plt.xticks(fontsize=12)
#         plt.legend()

#         # Display the plot
#         plt.show()

#     return maxcpa_matrix, correlation

def hd_cpa_last_round(plaintext,num_traces,c,target_byte=0,r1=9,r2=8):
#     c = AES(keys)
    
    leakage = np.empty(num_traces, dtype=object)
#     print(r1,r2)
    for i in range(num_traces):
        a,b,r_1,r_2 = c.encrypt(plaintext[i],round_1=r1,round_2=r2)
        if r1==10:
#             print("reached round 10")
            leakage[i] = bin(a[target_byte] ^ b[target_byte]).count('1')
        elif r1!=10:
#             print("i dont listen")
            leakage[i] = bin(r_1[target_byte] ^ r_2[target_byte]).count('1')
        
    return leakage

def hd_byte_cpa_last_round(plaintext,num_traces,c,target_byte=0,r1=9,r2=8):
#     c = AES(keys)
    
    leakage = np.empty(num_traces, dtype=object)
#     print(r1,r2)
    for i in range(num_traces):
        a,b,r_1,r_2 = c.encrypt(plaintext[i],round_1=r1,round_2=r2)
        if r1==10:
#             print("reached round 10")
            leakage[i] = a[target_byte] ^ b[target_byte]
        elif r1!=10:
#             print("i dont listen")
            leakage[i] =r_1[target_byte] ^ r_2[target_byte]
        
    return leakage


def hw_cpa_last_round(plaintext,num_traces,c,target_byte=0,r1=10,r2=8):
#     c = AES(keys)
    
    leakage = np.empty(num_traces, dtype=object)
#     print(r1,r2)
    for i in range(num_traces):
        a,b,r_1,_ = c.encrypt(plaintext[i],round_1=r1,round_2=r2)
        if r1==10:
#             print("reached round 10")
            leakage[i] = bin(a[target_byte]).count('1')
        elif r1!=10:
#             print("i dont listen")
#             k10 = decode_9(c._Ke[10])
            leakage[i] = bin(r_1[target_byte]).count('1')
        
    return leakage

def byte_snr_last_round(plaintext,c,num_traces,target_byte=0):
#     c = AES(keys)

    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        _,leakage[i] = c.encrypt(plaintext[i])
#         print(leakage[i])
    return [row[target_byte] for row in leakage]

In [3]:




from sklearn.kernel_approximation import Nystroem  # outside the function, once



import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
from scipy.stats import pearsonr


import numpy as np
from scipy.spatial.distance import cdist
from numpy.linalg import eigh
import matplotlib.pyplot as plt

def median_heuristic(X):
    """
    Compute sigma using the median heuristic.
    
    Parameters
    ----------
    X : ndarray of shape (n, d)
        Input data, n samples in d dimensions.
    
    Returns
    -------
    sigma : float
        Median heuristic bandwidth parameter.
    """
    X = np.asarray(X)
    n = X.shape[0]

    # Compute pairwise squared distances
    diff = X[:, None, :] - X[None, :, :]
    D2 = np.sum(diff**2, axis=2)

    # Take only the upper triangle (i<j) to avoid duplicates/zeros
    iu = np.triu_indices(n, k=1)
    sigma = np.sqrt(np.median(D2[iu]))
    return sigma

def effective_rank(K):
    """Effective rank = (sum λ)^2 / sum λ^2, using symmetric eigendecomp."""
    # eigh for symmetric matrices; ensure numerical non-negativity
    w = eigh(0.5*(K+K.T), UPLO='L')[0]
    w = np.clip(w, 0.0, None)
    s1 = np.sum(w)
    s2 = np.sum(w**2)
    return (s1**2 / s2) if s2 > 0 else 0.0

def moving_average(x, k=3):
    """Simple centered moving average (odd k). Falls back to same length."""
    if k <= 1 or k % 2 == 0:
        return x
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode='edge')
    kernel = np.ones(k) / k
    return np.convolve(xp, kernel, mode='valid')


def calculate_MI_EM_percell(traces,labels, alpha=1.01, normalize = False,nystrom = False,m_l=500, sigma_value=2.9 ):
    print('running modified code for experiments')
    
    
    
    variances = np.var(traces, axis=0)

    mask = variances >0.00001
    traces = traces[:, mask]
    observations_at_time = traces
    o = traces

    #find the range of the sigma

    X = traces
    m = X.shape[0]

    # 1) pairwise squared distances
    D2 = cdist(X, X, metric='euclidean')**2

    # 2) median heuristic for base sigma
    upper = D2[np.triu_indices(m, k=1)]
#     sigma0 = np.sqrt(np.median(upper)) if upper.size > 0 else 1.0
    sigma0 = np.sqrt(0.5*np.median(upper)) if upper.size > 0 else 1.0


    # 3) sweep σ

    span_low = 1/3
    span_high = 10
    n_sigma = 20


    sigmas = np.logspace(np.log10(sigma0*span_low),
                         np.log10(sigma0*span_high),
                         num=n_sigma)

    # eff_ranks = np.zeros_like(sigmas)
    # MI_vals   = np.zeros_like(sigmas)

    eff_ranks = []
    MI_vals = []
    valid_sigmas =[]
    Hx_ = []
    Hxy_ =[]


    for i in range(n_sigma):


        sigma_x = sigmas[i]


        h = np.array(labels, dtype=int)

        x = torch.tensor(o, dtype=torch.float32)
        y = torch.tensor(h, dtype=torch.float32)




        mio,Hx, Hxy, kx = calculate_MI(x , y,1.01, sigma_x,1,False)   
        Hy = Hxy + mio -Hx
#         if (Hxy > max(Hx,Hy) and mio < min(Hx,Hy) and mio>0 ):
        if (Hxy >= max(Hx,Hy) and mio>=0 ):
            eff_ranks.append(effective_rank(kx))
            MI_vals.append(mio)
            valid_sigmas.append(sigma_x)
            Hx_.append(Hx)
            Hxy_.append(Hxy)
            


    eff_ranks = np.array(eff_ranks)
    MI_vals = np.array(MI_vals)
    sigmas = np.array(valid_sigmas)
    smooth_k = 5
    
#     if len(eff_ranks) == 0 or len(MI_vals) == 0:
#         return np.nan

    rank_smooth = moving_average(eff_ranks, k=smooth_k)
    

    MI_smooth   = moving_average(MI_vals,   k=smooth_k)
#     if len(rank_smooth) < 2:
#         print("Not enough valid σ values to find elbow — using median σ instead.")
#         elbow_idx = len(sigmas) // 2
#     else:
#         eps = 1e-12
#         d1 = np.gradient(np.log(rank_smooth + eps))
#         d2 = np.gradient(d1)
#         elbow_idx = int(np.argmin(d2))
    eps = 1e-12
    d1 = np.gradient(np.log(rank_smooth + eps))
    d2 = np.gradient(d1)
    elbow_idx = int(np.argmin(d2))
    mi_peak_idx = int(np.argmax(MI_smooth))
    sigma_opt_idx = min(elbow_idx, mi_peak_idx)
    sigma_opt = float(sigmas[sigma_opt_idx])
    
    
    Hx_nystrom = renyi_entropy_n(x,sigma_opt,1.01, 280) 
    
    return Hx_nystrom

    

    
    
#     if(nystrom==False):
#         mio,Hx, Hxy, kx = calculate_MI(x , y,1.01, sigma_x,sigma_y,False)
#         print(kx)
#     else:
#         mio,Hx, Hxy = calculate_MI_nystrom(x, y, alpha=1.01, sigma_x=sigma_opt, m_landmarks=m_l, seed=42, ridge=1e-8, return_intermediates=False)
    
    
    


        
    



def calculate_gram_mat(x, sigma):
    """calculate gram matrix for variables x
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
    Returns:
        Gram matrix (N,N)
    """
    x = x.view(x.shape[0],-1)
    instances_norm = torch.sum(x**2,-1).reshape((-1,1))
    dist= -2*torch.mm(x,x.t()) + instances_norm + instances_norm.t()
    return torch.exp(-dist /sigma)
def renyi_entropy(x,sigma,alpha):
    """calculate entropy for single variables x (Eq.(9) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
        alpha:  alpha value of renyi entropy
    Returns:
        renyi alpha entropy of x.
    """
    k = calculate_gram_mat(x,sigma)
    k = k/torch.trace(k)
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
    eigv = torch.abs(eigv)
    eig_pow = eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy, k

def renyi_entropy_n(x,sigma,alpha, landmark):
    """calculate entropy for single variables x (Eq.(9) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
        alpha:  alpha value of renyi entropy
    Returns:
        renyi alpha entropy of x.
    """
    #k = calculate_gram_mat(x,sigma)

    # nystrom 
    
    feature_map = Nystroem(kernel='rbf', gamma=1.0/float(sigma), n_components=landmark)

    # line 2: get Φ (N × m) from Nyström
    Phi_np = feature_map.fit_transform(x.detach().cpu().numpy())

    # line 3: approximate Gram matrix K ≈ Φ Φᵀ in torch
    k = torch.from_numpy(Phi_np).to(x.device)
    k = k @ k.T
    
    #  nystrom 
    





    k = k/torch.trace(k)
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
    eigv = torch.abs(eigv)
    eig_pow = eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy




def k_cal(x):
    m = x.shape[0]
    classes = torch.unique(x)
    C = len(classes)
    L = torch.zeros((m, C), dtype=torch.float32)
    for c, class_label in enumerate(classes):
        idx = (x == class_label)
        n_c = idx.sum()
        if n_c > 0:
            L[idx, c] = 1.0 / torch.sqrt(n_c.float())
    # Compute Kx = L * L^T
    k = L @ L.T  # Matrix multiplication
    return k
def renyi_entropy_labels(x,sigma,alpha):
    """calculate entropy for single variables x (Eq.(9) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        sigma: kernel size of x (Gaussain kernel)
        alpha:  alpha value of renyi entropy
    Returns:
        renyi alpha entropy of x.
    """
    m = x.shape[0]
    classes = torch.unique(x)
    C = len(classes)
    L = torch.zeros((m, C), dtype=torch.float32)
    for c, class_label in enumerate(classes):
        idx = (x == class_label)
        n_c = idx.sum()
        if n_c > 0:
            L[idx, c] = 1.0 / torch.sqrt(n_c.float())
    # Compute Kx = L * L^T
    k = L @ L.T  # Matrix multiplication
    # Normalize trace to 1
    k = k / torch.trace(k)
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
    eigv = torch.abs(eigv)
    eig_pow = eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy
def joint_entropy(x,y,s_x,s_y,alpha):
    """calculate joint entropy for random variable x and y (Eq.(10) in paper)
        Args:
        x: random variable with two dimensional (N,d).
        y: random variable with two dimensional (N,d).
        s_x: kernel size of x
        s_y: kernel size of y
        alpha:  alpha value of renyi entropy
    Returns:
        joint entropy of x and y.
    """
    x = calculate_gram_mat(x,s_x)
    # y = calculate_gram_mat(y,s_y)
    y = k_cal(y)
    k = torch.mul(x,y)
    # eps = 1e-5  # or tune this
    # k += eps * torch.eye(k.shape[0], device=k.device)
    k = k/torch.trace(k)
    eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
    eigv = torch.abs(eigv)
    eig_pow =  eigv**alpha
    entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
    return entropy
def calculate_MI(x,y,alpha,s_x,s_y,normalize):
    """calculate Mutual information between random variables x and y
    Args:
        x: random variable with two dimensional (N,d).
        y: random variable with two dimensional (N,d).
        s_x: kernel size of x
        s_y: kernel size of y
        normalize: bool True or False, noramlize value between (0,1)
    Returns:
        Mutual information between x and y (scale)
    """
    Hx , kx = renyi_entropy(x,sigma=s_x , alpha=alpha)
    # Hy = renyi_entropy(y,sigma=s_y , alpha=alpha)
    Hy = renyi_entropy_labels(y,sigma=s_y , alpha=alpha)

    Hxy= joint_entropy(x,y,s_x,s_y , alpha=alpha)
    if normalize:
        Ixy = Hx+Hy-Hxy
        Ixy = Ixy/(torch.max(Hx,Hy))
    else:
        Ixy = Hx+Hy-Hxy
    return Ixy,Hx, Hxy, kx



def get_sigma(dim, n, std):
    h = (0.9*std)/(n**1.5)
    return h*n**(-1/(4+dim))













In [ ]:
# #MI nystrom 
#old functions for MI - mohammad

# # import torch
# def standard_profiler(X, Y):
#     """
#     Replicates the C++ profiler behavior: 
#     - Finds the most informative feature using Pearson correlation.
#     - Groups samples by labels.
#     - Computes mean and standard deviation for each key hypothesis.
    
#     Parameters:
#         X (numpy.ndarray): Leakage traces, shape (n_traces, n_features).
#         Y (numpy.ndarray): Labels (S-Box outputs), shape (n_traces,).
        
#     Returns:
#         profile (numpy.ndarray): A (256, 2) array where:
#             - Row 0 contains mean values for each key hypothesis.
#             - Row 1 contains standard deviations for each key hypothesis.
#         samples (dict): A dictionary where keys are labels (0-255) and values are lists of leakage values.
#         best_feature_idx (int): The index of the most correlated feature.
#     """
    
#     n_traces, n_features = X.shape  # Get the number of traces and features
    
#     # Compute Pearson correlation for each feature
#     correlations = np.array([abs(pearsonr(X[:, i], Y)[0]) for i in range(n_features)])
    
#     # Find the feature with the highest correlation
#     best_feature_idx = np.argmax(correlations)
   

#     return best_feature_idx

# # def calculate_gram_mat(x, sigma):
# #     """calculate gram matrix for variables x
# #         Args:
# #         x: random variable with two dimensional (N,d).
# #         sigma: kernel size of x (Gaussain kernel)
# #     Returns:
# #         Gram matrix (N,N)
# #     """
# #     x = x.view(x.shape[0],-1)
# #     instances_norm = torch.sum(x**2,-1).reshape((-1,1))
# #     dist= -2*torch.mm(x,x.t()) + instances_norm + instances_norm.t()
# #     return torch.exp(-dist /sigma)

# # def renyi_entropy(x,sigma,alpha):
    
# #     """calculate entropy for single variables x (Eq.(9) in paper)
# #         Args:
# #         x: random variable with two dimensional (N,d).
# #         sigma: kernel size of x (Gaussain kernel)
# #         alpha:  alpha value of renyi entropy
# #     Returns:
# #         renyi alpha entropy of x. 
# #     """
    
# #     k = calculate_gram_mat(x,sigma)
# #     # eps = 1e-5 # or tune this
# #     # k += eps * torch.eye(k.shape[0], device=k.device)

# #     k = k/torch.trace(k) 
# #     eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part

# #     eigv = torch.abs(eigv)  
# #     eig_pow = eigv**alpha
# #     entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
# #     return entropy

# # def joint_entropy(x,y,s_x,s_y,alpha):
    
# #     """calculate joint entropy for random variable x and y (Eq.(10) in paper)
# #         Args:
# #         x: random variable with two dimensional (N,d).
# #         y: random variable with two dimensional (N,d).
# #         s_x: kernel size of x
# #         s_y: kernel size of y
# #         alpha:  alpha value of renyi entropy
# #     Returns:
# #         joint entropy of x and y. 
# #     """
    
# #     x = calculate_gram_mat(x,s_x)
# #     y = calculate_gram_mat(y,s_y)
# #     k = torch.mul(x,y)
    
# #     # eps = 1e-5  # or tune this
# #     # k += eps * torch.eye(k.shape[0], device=k.device)

    

# #     k = k/torch.trace(k)
# #     eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part

# #     eigv = torch.abs(eigv)
# #     eig_pow =  eigv**alpha
# #     entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
# #     return entropy

# # def calculate_MI(x,y,alpha,s_x,s_y,normalize):
    
# #     """calculate Mutual information between random variables x and y

# #     Args:
# #         x: random variable with two dimensional (N,d).
# #         y: random variable with two dimensional (N,d).
# #         s_x: kernel size of x
# #         s_y: kernel size of y
# #         normalize: bool True or False, noramlize value between (0,1)
# #     Returns:
# #         Mutual information between x and y (scale)

# #     """
# #     Hx = renyi_entropy(x,sigma=s_x , alpha=alpha)
# #     Hy = renyi_entropy(y,sigma=s_y , alpha=alpha)
# #     Hxy = joint_entropy(x,y,s_x,s_y , alpha=alpha)
# #     if normalize:
# #         Ixy = Hx+Hy-Hxy
# #         Ixy = Ixy/(torch.max(Hx,Hy))
# #     else:
# #         Ixy = Hx+Hy-Hxy
# #     return Ixy

# def calculate_gram_mat(x, sigma):
#     """calculate gram matrix for variables x
#         Args:
#         x: random variable with two dimensional (N,d).
#         sigma: kernel size of x (Gaussain kernel)
#     Returns:
#         Gram matrix (N,N)
#     """
#     x = x.view(x.shape[0],-1)
#     instances_norm = torch.sum(x**2,-1).reshape((-1,1))
#     dist= -2*torch.mm(x,x.t()) + instances_norm + instances_norm.t()
#     return torch.exp(-dist /sigma)
# def renyi_entropy(x,sigma,alpha):
#     """calculate entropy for single variables x (Eq.(9) in paper)
#         Args:
#         x: random variable with two dimensional (N,d).
#         sigma: kernel size of x (Gaussain kernel)
#         alpha:  alpha value of renyi entropy
#     Returns:
#         renyi alpha entropy of x.
#     """
#     k = calculate_gram_mat(x,sigma)
#     k = k/torch.trace(k)
#     eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
#     eigv = torch.abs(eigv)
#     eig_pow = eigv**alpha
#     entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
#     return entropy
# def k_cal(x):
#     m = x.shape[0]
#     classes = torch.unique(x)
#     C = len(classes)
#     L = torch.zeros((m, C), dtype=torch.float32)
#     for c, class_label in enumerate(classes):
#         idx = (x == class_label)
#         n_c = idx.sum()
#         if n_c > 0:
#             L[idx, c] = 1.0 / torch.sqrt(n_c.float())
#     # Compute Kx = L * L^T
#     k = L @ L.T  # Matrix multiplication
#     return k
# def renyi_entropy_labels(x,sigma,alpha):
#     """calculate entropy for single variables x (Eq.(9) in paper)
#         Args:
#         x: random variable with two dimensional (N,d).
#         sigma: kernel size of x (Gaussain kernel)
#         alpha:  alpha value of renyi entropy
#     Returns:
#         renyi alpha entropy of x.
#     """
#     m = x.shape[0]
#     classes = torch.unique(x)
#     C = len(classes)
#     L = torch.zeros((m, C), dtype=torch.float32)
#     for c, class_label in enumerate(classes):
#         idx = (x == class_label)
#         n_c = idx.sum()
#         if n_c > 0:
#             L[idx, c] = 1.0 / torch.sqrt(n_c.float())
#     # Compute Kx = L * L^T
#     k = L @ L.T  # Matrix multiplication
#     # Normalize trace to 1
#     k = k / torch.trace(k)
#     eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
#     eigv = torch.abs(eigv)
#     eig_pow = eigv**alpha
#     entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
#     return entropy
# def joint_entropy(x,y,s_x,s_y,alpha):
#     """calculate joint entropy for random variable x and y (Eq.(10) in paper)
#         Args:
#         x: random variable with two dimensional (N,d).
#         y: random variable with two dimensional (N,d).
#         s_x: kernel size of x
#         s_y: kernel size of y
#         alpha:  alpha value of renyi entropy
#     Returns:
#         joint entropy of x and y.
#     """
#     x = calculate_gram_mat(x,s_x)
#     # y = calculate_gram_mat(y,s_y)
#     y = k_cal(y)
#     k = torch.mul(x,y)
#     # eps = 1e-5  # or tune this
#     # k += eps * torch.eye(k.shape[0], device=k.device)
#     k = k/torch.trace(k)
#     eigv, _ = torch.linalg.eigh(k, UPLO='U')  # or 'L' if you want the lower part
#     eigv = torch.abs(eigv)
#     eig_pow =  eigv**alpha
#     entropy = (1/(1-alpha))*torch.log2(torch.sum(eig_pow))
#     return entropy
# def calculate_MI(x,y,alpha,s_x,s_y,normalize):
#     """calculate Mutual information between random variables x and y
#     Args:
#         x: random variable with two dimensional (N,d).
#         y: random variable with two dimensional (N,d).
#         s_x: kernel size of x
#         s_y: kernel size of y
#         normalize: bool True or False, noramlize value between (0,1)
#     Returns:
#         Mutual information between x and y (scale)
#     """
#     Hx = renyi_entropy(x,sigma=s_x , alpha=alpha)
#     # Hy = renyi_entropy(y,sigma=s_y , alpha=alpha)
#     Hy = renyi_entropy_labels(y,sigma=s_y , alpha=alpha)

#     Hxy= joint_entropy(x,y,s_x,s_y , alpha=alpha)
#     if normalize:
#         Ixy = Hx+Hy-Hxy
#         Ixy = Ixy/(torch.max(Hx,Hy))
#     else:
#         Ixy = Hx+Hy-Hxy
#     return Ixy,Hx, Hxy



# def get_sigma(dim, n, std):
#     h = (0.9*std)/(n**1.5)
#     return h*n**(-1/(4+dim))





# # def calculate_MI_EM(traces,labels,num_columns=100, alpha=1.01, normalize = False,multiplier=1):
# # #     # Set sigma values (choose based on data distribution)
# # #     sigma_x = torch.std(x) / 2  # Example heuristic for sigma
# # #     sigma_y = torch.std(y) / 2
# #     # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# #     mi=[]
# #     for i in range(num_columns):   
        
# #         x = torch.tensor(np.round(multiplier*traces[:,i]), dtype=torch.float32)  # Reshape to (N,1)
# #         y = torch.tensor(list(labels), dtype=torch.float32) # Reshape to (N,1)
# #         # x = x.to(device)
# #         # y = y.to(device)
# #         sigma_x = get_sigma(1,len(labels),torch.std(x) )
# #         sigma_y = get_sigma(1,len(labels),torch.std(y) )

# #         mi.append(calculate_MI(x , y , alpha,sigma_x,sigma_y,normalize))
        
# #     return np.array([val.detach().cpu().numpy() for val in mi])

# def median_heuristic(X):
#     """
#     Compute sigma using the median heuristic.
    
#     Parameters
#     ----------
#     X : ndarray of shape (n, d)
#         Input data, n samples in d dimensions.
    
#     Returns
#     -------
#     sigma : float
#         Median heuristic bandwidth parameter.
#     """
#     X = np.asarray(X)
#     n = X.shape[0]

#     # Compute pairwise squared distances
#     diff = X[:, None, :] - X[None, :, :]
#     D2 = np.sum(diff**2, axis=2)

#     # Take only the upper triangle (i<j) to avoid duplicates/zeros
#     iu = np.triu_indices(n, k=1)
#     sigma = np.sqrt(np.median(D2[iu]))
#     return sigma


# def calculate_MI_EM_percell(traces,labels, alpha=1.01, normalize = False,nystrom = False,m_l=500):
    
    
#     variances = np.var(traces, axis=0)

#     mask = variances >0.001
#     traces = traces[:, mask]
#     observations_at_time = traces
#     o = traces
    
#     sigma_x = 2.97


#     h = np.array(labels, dtype=int)

#     x = torch.tensor(o, dtype=torch.float32)
#     y = torch.tensor(h, dtype=torch.float32)


# #     sigma_x = get_sigma(traces.shape[1],1000,torch.std(x) )
#     sigma_y = get_sigma(1,1000,torch.std(y) )
#     if(nystrom==False):
#         mio,Hx, Hxy = calculate_MI(x , y,1.01, sigma_x,sigma_y,False)
#     else:
#         mio,Hx, Hxy = calculate_MI_nystrom(x, y, alpha=1.01, sigma_x=sigma_x, m_landmarks=m_l, seed=42, ridge=1e-8, return_intermediates=False)
    
    
    


        
#     return np.array(mio),np.array(Hx), np.array(Hxy)
# # # ----------------------------
# # Kernel utilities (your convention)
# # ----------------------------
# def _pairwise_sq_dists(X, Y):
#     # X: (N,d), Y: (M,d) -> D: (N,M) with ||x - y||^2
#     Xn = (X * X).sum(dim=1, keepdim=True)      # (N,1)
#     Yn = (Y * Y).sum(dim=1, keepdim=True).T    # (1,M)
#     return torch.clamp(Xn - 2 * X @ Y.T + Yn, min=0.0)

# def _rbf_block(X, Y, sigma):
#     # Your convention: exp(-dist / sigma)
#     D = _pairwise_sq_dists(X, Y)
#     return torch.exp(-D / sigma)

# # ----------------------------
# # Label kernel via normalized indicator matrix
# # Ky = L L^T, where L[i,c] = 1/sqrt(n_c) if y_i == c else 0
# # ----------------------------
# def _label_L_matrix(y):
#     """
#     y: (N,) integer labels (tensor)
#     Returns: L in R^{N x C}, where C = #classes in y
#     """
#     device = y.device
#     dtype = torch.float32
#     y = y.view(-1)
#     classes = torch.unique(y)
#     C = classes.numel()
#     N = y.shape[0]

#     L = torch.zeros((N, C), dtype=dtype, device=device)
#     for c, class_label in enumerate(classes):
#         idx = (y == class_label)
#         n_c = idx.sum()
#         if n_c > 0:
#             L[idx, c] = 1.0 / torch.sqrt(n_c.float())
#     return L, classes

# # ----------------------------
# # Nyström for a given X with RBF kernel
# # ----------------------------
# def nystrom_features_rbf(X, m, sigma, seed=0, ridge=1e-8, given_indices=None):
#     """
#     Build Nyström features Phi for X using m landmarks under RBF kernel.

#     X: (N,d)
#     m: #landmarks
#     sigma: kernel scale (your convention)
#     seed: for reproducible landmark sampling (ignored if given_indices is provided)
#     ridge: small jitter for stability on W
#     given_indices: optional pre-chosen landmark indices (1D LongTensor)

#     Returns:
#       Phi: (N, m) such that K ≈ Phi @ Phi.T
#       idx: landmark indices (LongTensor, shape (m,))
#     """
#     N = X.shape[0]
#     assert m <= N, "m must be <= number of samples"

#     if given_indices is None:
#         g = torch.Generator(device=X.device)
#         g.manual_seed(seed)
#         idx = torch.randperm(N, generator=g, device=X.device)[:m]
#     else:
#         idx = given_indices
#         m = idx.numel()
#         assert m <= N

#     Z = X[idx]                               # (m,d)
#     C = _rbf_block(X, Z, sigma)              # (N,m)
#     W = _rbf_block(Z, Z, sigma)              # (m,m)

#     # Stabilize and invert sqrt(W)
#     W_stab = W + ridge * torch.eye(m, device=X.device, dtype=X.dtype)
#     evals, evecs = torch.linalg.eigh(W_stab)
#     evals = torch.clamp(evals, min=1e-12)
#     Winvsqrt = evecs @ torch.diag(evals.rsqrt()) @ evecs.T  # (m,m)

#     Phi = C @ Winvsqrt                       # (N,m)
#     return Phi, idx

# # ----------------------------
# # Nyström for label kernel Ky = L L^T using the SAME indices
# # ----------------------------
# def nystrom_features_labels(y, landmark_idx, ridge=1e-8):
#     """
#     Build Nyström features for the label kernel using the SAME landmark indices.

#     y: (N,) integer labels
#     landmark_idx: (m,) LongTensor, indices chosen from X (we reuse for Y)
#     ridge: stability term

#     Returns:
#       Phi_y: (N, m) such that Ky ≈ Phi_y @ Phi_y.T
#     """
#     L, _ = _label_L_matrix(y)        # (N, C)
#     Lz = L[landmark_idx]             # (m, C)

#     # For Ky = L L^T, we can get:
#     # C_y = Ky[:, Z] = L @ Lz^T  => (N,m)
#     # W_y = Ky[Z, Z] = Lz @ Lz^T => (m,m)
#     C_y = L @ Lz.T                   # (N, m)
#     W_y = Lz @ Lz.T                  # (m, m)

#     # Stabilize and invert sqrt(W_y)
#     m = W_y.shape[0]
#     W_stab = W_y + ridge * torch.eye(m, device=W_y.device, dtype=W_y.dtype)
#     evals, evecs = torch.linalg.eigh(W_stab)
#     evals = torch.clamp(evals, min=1e-12)
#     Winvsqrt = evecs @ torch.diag(evals.rsqrt()) @ evecs.T

#     Phi_y = C_y @ Winvsqrt           # (N,m)
#     return Phi_y

# # ----------------------------
# # Entropy helpers using Nyström features
# # ----------------------------
# def _renyi_entropy_from_phi(Phi, alpha):
#     """
#     Compute matrix-based Rényi entropy of K ≈ Phi Phi^T
#     WITHOUT forming K explicitly. Uses the spectrum of G = Phi^T Phi.

#     We follow the paper's normalization: K <- K / trace(K).
#     Since trace(Phi Phi^T) = ||Phi||_F^2 = trace(G), we can normalize
#     by dividing eigenvalues of G by their sum.

#     Returns H_alpha in bits (log2).
#     """
#     # G = Phi^T Phi (m x m), SPSD
#     G = Phi.T @ Phi
#     # Eigen on small matrix
#     evals, _ = torch.linalg.eigh(G)
#     evals = torch.clamp(evals, min=0.0)

#     s = evals.sum()
#     if s <= 0:
#         # Degenerate case: return 0
#         return torch.tensor(0.0, device=Phi.device, dtype=Phi.dtype)

#     lam = evals / s  # trace-normalized eigenvalues (sum to 1)
#     # Avoid zeros in pow/log for numerical stability
#     lam = torch.clamp(lam, min=1e-30)

#     if abs(alpha - 1.0) < 1e-8:
#         # Shannon limit (optional): use l'Hôpital / log with lam*log lam
#         H = -torch.sum(lam * torch.log2(lam))
#     else:
#         H = (1.0 / (1.0 - alpha)) * torch.log2(torch.sum(lam ** alpha))
#     return H

# def _build_K_from_phi(Phi):
#     # Build K ≈ Phi Phi^T (N x N); enforce symmetry
#     K = Phi @ Phi.T
#     return 0.5 * (K + K.T)

# def _renyi_entropy_from_full_K(K, alpha):
#     # Normalize trace(K) = 1, then use eigenvalues
#     K = K / torch.trace(K)
#     evals, _ = torch.linalg.eigh(K)          # symmetric
#     evals = torch.clamp(evals, min=0.0)
#     evals = torch.clamp(evals, min=1e-30)
#     if abs(alpha - 1.0) < 1e-8:
#         H = -torch.sum(evals * torch.log2(evals))
#     else:
#         H = (1.0 / (1.0 - alpha)) * torch.log2(torch.sum(evals ** alpha))
#     return H

# # ----------------------------
# # Public API: MI with Nyström (same landmarks for X and Y)
# # ----------------------------
# def calculate_MI_nystrom(
#     X, y, alpha, sigma_x, m_landmarks, seed=0, ridge=1e-8, return_intermediates=False
# ):
#     """
#     Mutual information I_alpha(X;Y) using Nyström approximations.

#     X: (N,d) float tensor
#     y: (N,)  int tensor (labels)
#     alpha: Rényi order (>0, alpha!=1 recommended; alpha≈1 for Shannon limit)
#     sigma_x: RBF scale for X kernel (your convention)
#     m_landmarks: number of Nyström landmarks (<= N)
#     seed: for reproducible landmark choice on X
#     ridge: stability for W^{-1/2}
#     return_intermediates: if True, returns a dict of pieces

#     Returns:
#       Ixy: scalar tensor (bits)
#       (optionally) details dict
#     """
#     # 1) Nyström features for X; get indices we will reuse for Y
#     Phi_x, idx = nystrom_features_rbf(
#         X, m=m_landmarks, sigma=sigma_x, seed=seed, ridge=ridge, given_indices=None
#     )

#     # 2) Nyström features for Y using SAME idx
#     Phi_y = nystrom_features_labels(y, landmark_idx=idx, ridge=ridge)

#     # 3) Entropies H(X) and H(Y) from small spectra
#     Hx = _renyi_entropy_from_phi(Phi_x, alpha=alpha)
#     Hy = _renyi_entropy_from_phi(Phi_y, alpha=alpha)

#     # 4) Joint entropy via Hadamard product Kxy = Kx ⊙ Ky
#     #    Build Kx and Ky from features, then elementwise product
#     Kx = _build_K_from_phi(Phi_x)   # (N,N)
#     Ky = _build_K_from_phi(Phi_y)   # (N,N)
#     Kxy = Kx * Ky                   # Hadamard product
#     Hxy = _renyi_entropy_from_full_K(Kxy, alpha=alpha)

#     Ixy = Hx + Hy - Hxy

#     if return_intermediates:
#         details = {
#             "Hx": Hx, "Hy": Hy, "Hxy": Hxy,
#             "idx": idx,
#             "Phi_x": Phi_x, "Phi_y": Phi_y,
#             "Kx": Kx, "Ky": Ky, "Kxy": Kxy
#         }
#         return Ixy, details
#     return Ixy,Hx, Hxy

# # ----------------------------
# # (Optional) exact baselines using your original code style
# # ----------------------------
# def calculate_gram_mat_exact(x, sigma):
#     x = x.view(x.shape[0], -1)
#     instances_norm = torch.sum(x ** 2, -1).reshape((-1, 1))
#     dist = -2 * torch.mm(x, x.t()) + instances_norm + instances_norm.t()
#     return torch.exp(-dist / sigma)

# def renyi_entropy_exact_X(x, sigma, alpha):
#     K = calculate_gram_mat_exact(x, sigma)
#     return _renyi_entropy_from_full_K(K, alpha)

# def renyi_entropy_exact_labels(y, alpha):
#     L, _ = _label_L_matrix(y)
#     K = L @ L.T
#     return _renyi_entropy_from_full_K(K, alpha)

# def joint_entropy_exact(x, y, s_x, alpha):
#     Kx = calculate_gram_mat_exact(x, s_x)
#     L, _ = _label_L_matrix(y)
#     Ky = L @ L.T
#     Kxy = Kx * Ky
#     return _renyi_entropy_from_full_K(Kxy, alpha)









# deprecated functions


In [ ]:
      
        
# def Grid_Tracing(X_range,Y_range,X_number_of_step,Y_number_of_step,X,Y,Z,interface,sco,number_of_traces):
#     cordinate_traces={}
#     X_moment=0
#     Y_moment=0
#     traces = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#     cordinate = (X_moment, Y_moment)
#     cordinate_traces[cordinate] = traces
#     X_start_position=X.get_actual_position()
#     Y_start_position=Y.get_actual_position()
#     print(f"Starting Postion - ({X_start_position},{Y_start_position})")
#     while Y_moment<=Y_number_of_step:
#         X_initial_position=X.get_actual_position()
#         Y_initial_position=Y.get_actual_position()

#         for _ in range(X_number_of_step):
#             X.move_by(X_range)
#             print(f'Moving X to {X_initial_position + X_range}')

#             while X.get_actual_position() != X_initial_position + X_range:


#                 time.sleep(0.1)
#             traces = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#             X_moment += 1
#             cordinate = (X_moment, Y_moment)
#             cordinate_traces[cordinate] = traces
#             X_initial_position=X.get_actual_position()

#         if Y_moment==Y_number_of_step:
#             break
#         Y.move_by(Y_range)
#         Y_moment+=1
#         print(f'Moving Y to {Y_initial_position + Y_range}')
#         while Y.get_actual_position() != Y_initial_position + Y_range:

#             time.sleep(0.1)
#         cordinate = (X_moment, Y_moment)
#         cordinate_traces[cordinate] = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#         Y_initial_position=Y.get_actual_position()

#         for _ in range(X_number_of_step):
#             X.move_by(-X_range)
#             print(f'Moving X to {X_initial_position - X_range}')
#             while X.get_actual_position() != X_initial_position - X_range:

#                 time.sleep(0.1)
#             traces = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#             X_moment -= 1
#             cordinate = (X_moment, Y_moment)
#             cordinate_traces[cordinate] = traces
#             X_initial_position=X.get_actual_position()
#         if Y_moment==Y_number_of_step:
#             break
#         # Move down 1 step
#         Y.move_by(Y_range)
#         Y_moment += 1
#         print(f'Moving Y to {Y_initial_position + Y_range}')
#         while Y.get_actual_position() != Y_initial_position + Y_range:
#             time.sleep(0.1)

#         cordinate = (X_moment, Y_moment)
#         cordinate_traces[cordinate] = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#         Y_initial_position=Y.get_actual_position()
    
#     X.move_to(X_start_position)
#     while X.get_actual_position() != X_start_position:
#         time.sleep(0.1)
#     Y.move_to(Y_start_position)
#     while Y.get_actual_position() != Y_start_position:
#         time.sleep(0.1)
#     print(f"Final Postion - ({X.get_actual_position()},{Y.get_actual_position()})")

#     return cordinate_traces


def plot_SNR_heatmap_byte(test, pt_exp, num, target_byte=0, grid_size=5):
    """
    Calculate Signal-to-Noise Ratio (SNR) for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that provides methods to retrieve traces.
    - pt_exp: Experiment data handler containing keys and plaintexts.
    - num: Number of traces to consider.
    - target_byte: The target byte of the AES S-box (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    
    Returns:
    - CEMA_values_rotated: The rotated matrix of SNR values.
    - Log-transformed CEMA values: Log-transformed SNR values for better visualization.
    """
    # Load keys and plaintexts from the experiment data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Initialize the CEMA values matrix
    CEMA_values = np.zeros((grid_size, grid_size))

    # Calculate the SNR labels based on the chosen target byte
    labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    labelsUnique = np.unique(labels)

    # Iterate over the grid and compute the SNR for each point
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {k: [] for k in labelsUnique}  # Initialize dictionary to hold sorted labels
            print(f"Processing grid point ({i}, {j})")

            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)

            # Organize the traces based on the labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # Calculate the SNR for the current grid point
            snr_value = signal_to_noise_ratio(sorted_labels)

            # Store the maximum SNR value for this grid point
            CEMA_values[i, j] = np.nanmax(np.abs(snr_value))

    # Rotate the CEMA values matrix
    CEMA_values_rotated = np.rot90(CEMA_values, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Plot the heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title to the plot
    plt.title("Heatmap of SNR Byte")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()

    # Return the rotated CEMA values and their log-transformed values
    return CEMA_values_rotated, 10 * np.log10(CEMA_values_rotated)

def plot_SNR_heatmap_hw_byte(test, pt_exp, num, target_byte=0, grid_size=5):
    """
    Calculate the Hamming Weight-based SNR for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - pt_exp: Experiment data handler containing keys and plaintexts.
    - num: Number of traces to consider.
    - target_byte: The target byte of the AES S-box (default is 0).
    - grid_size: The size of the grid (default is 5x5).
    
    Returns:
    - CEMA_values_rotated: The rotated matrix of SNR values.
    - Log-transformed CEMA values: Log-transformed SNR values for better visualization.
    """
    # Load keys and plaintexts from the experiment data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)

    # Initialize matrices for storing SNR values and guesses
    CEMA_values = np.zeros((grid_size, grid_size))
    CEMA_guesses = np.zeros((grid_size, grid_size))  # Not used currently, can be useful for future extensions

    # Calculate SNR labels using the Hamming Weight leakage model
    labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    labelsUnique = np.unique(labels)

    # Iterate over the grid to compute SNR for each point
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {k: [] for k in labelsUnique}  # Initialize the dictionary for sorted labels
            print(f"Processing grid point ({i}, {j})")

            traces = test.get_dataset(f"random_{i}_{j}").read_data(0, num)

            # Organize traces based on the labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # Calculate the SNR for the current grid point
            snr_value = signal_to_noise_ratio(sorted_labels)

            # Store the maximum SNR value for this grid point
            CEMA_values[i, j] = np.nanmax(np.abs(snr_value))

    # Rotate the CEMA values matrix
    CEMA_values_rotated = np.rot90(CEMA_values, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title to the plot
    plt.title("Heatmap of SNR Byte HW")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()

    # Return the rotated CEMA values and their log-transformed values
    return CEMA_values_rotated, 10 * np.log10(CEMA_values_rotated)


def plot_CEMA_traces_idk(traces, pt_exp, num, target_byte=0, x=0, y=0, div=10, visualize_correct = True):
    """
    Generate a CPA correlation plot comparing the correct key vs. wrong keys over increasing trace counts.

    This function performs Correlation Power Analysis (CPA) across different numbers of traces, 
    visualizing how the correct key and wrong keys' correlation evolve.

    Parameters:
    - test: An object that provides trace datasets.
    - pt_exp: Experiment data handler providing plaintext and key datasets.
    - num: Total number of traces to analyze.
    - target_byte: The target byte index in the key (default is 0).
    - x, y: Grid position for selecting the dataset.
    - div: Step size for processing traces in intervals.

    Returns:
    - maxcpa_matrix: A matrix storing the maximum CPA correlation values for all 256 key guesses.
    """

    # Load key and plaintext data
    keys = pt_exp.get_dataset("keys").read_data(0, num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0, num)
    correct_key = keys[0][target_byte]
    # Load power traces for the selected grid position
#     traces = test.get_dataset(f"random_{x}_{y}").read_data(0, num)

    # Initialize a matrix to store max CPA values across different key guesses
    maxcpa_matrix = np.zeros((int(num / div), 256))

    iterations = 1
    for i in trange(1, num):
        if i % div == 0:
            if visualize_correct:
                k = correct_key
                leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)

                # Compute Pearson correlation
                correlation = pearson_correlation(leakage, traces[:i])

                # Store max correlation value for this key guess
                maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))
            else:
                for k in range(256):
                    # Compute leakage model using Hamming weight
                    leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)

                    # Compute Pearson correlation
                    correlation = pearson_correlation(leakage, traces[:i])

                    # Store max correlation value for this key guess
                    maxcpa_matrix[int(i / div), k] = np.nanmax(np.abs(correlation))

                iterations += 1

    # Debugging: Check matrix shape
    print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

    if visualize_correct:
        xp =  np.arange(2,  maxcpa_matrix.shape[0])
    else:
        xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))

    # Plotting
    plt.figure(figsize=(10, 6))

    if maxcpa_matrix.shape[0] > 2:
        # Plot the statistical threshold
        plt.plot(xp, (abs(4) / np.sqrt(xp * div)) * np.ones_like(xp), 
                 color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

        # Plot CPA correlations for all 256 key hypotheses
        if visualize_correct:
            plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, int(correct_key)], color="red",
                     alpha=0.9, linewidth=1.5, label="Correct key")
        else:
            for i in range(256):
                if i == correct_key:  # Assuming 43 is the correct key
                    plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red",
                             alpha=0.9, linewidth=1.5, label="Correct key")
                else:
                    plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey",
                             alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")

    # Configure plot labels and title
    plt.xlabel(f"No. of traces × {div}", fontsize=14)
    plt.ylabel("Max CPA Value", fontsize=14)
    plt.title(f"Correlation Power Analysis", fontsize=18)
    plt.yticks(fontsize=12)
    plt.xticks(fontsize=12)
    plt.legend()

    # Display the plot
    plt.show()

    if visualize_correct:
        plt.figure(figsize=(10, 6))
        plt.plot(correlation)
        plt.xlabel(f"No. of traces × {div}", fontsize=14)
        plt.ylabel("Max CPA Value", fontsize=14)
        plt.title(f"Correlation Power Analysis", fontsize=18)
        plt.yticks(fontsize=12)
        plt.xticks(fontsize=12)
        plt.legend()

        # Display the plot
        plt.show()

    return maxcpa_matrix, correlation
def scapegoat_cpa_traces(traces, keys, plaintexts, target_byte):
    """
    Perform Correlation Power Analysis (CPA) on a specific key byte to guess the subkey.

    Parameters:
    - traces: The captured power traces.
    - keys: The actual secret keys corresponding to the traces.
    - plaintexts: The plaintexts used for the power analysis.
    - target_byte: The index of the target byte in the key to analyze.

    Returns:
    - best_guess: The best guess for the target key byte.
    - cpa_ref: The highest correlation value obtained for the target byte.
    """
    max_cpa = np.zeros(256)  # Store maximum CPA values for each possible subkey guess
    cpa_ref = 0  # Store highest correlation value for the target byte
    best_guess = 0  # Store best subkey guess for the target byte

    # Perform CPA attack for each possible subkey guess (0-255)
    for k in range(256):
        # Compute leakage model for each subkey guess
        leakage = leakage_model_hamming_weight(
            num_traces=len(plaintexts),
            plaintexts=plaintexts,
            subkey_guess=k,
            target_byte=target_byte
        )
        # Compute the Pearson correlation between the leakage and the traces
        correlation = pearson_correlation(leakage, traces)
        max_cpa[k] = np.nanmax(np.abs(correlation))  # Store the highest correlation for this guess

    # Find the best subkey guess and highest correlation value
    best_guess = np.argmax(max_cpa)
    cpa_ref = np.nanmax(max_cpa)

    return best_guess, cpa_ref

In [ ]:
# aes_round6_cpa.py
# Utilities to match aes_core.v signals and run CPA at round 6 (and last round baseline).
from typing import List, Tuple, Dict, Optional
import numpy as np

# ---------- AES primitives (byte-wise) ----------
SBOX = np.array([
    # 256 entries ...
    0x63,0x7c,0x77,0x7b,0xf2,0x6b,0x6f,0xc5,0x30,0x01,0x67,0x2b,0xfe,0xd7,0xab,0x76,
    0xca,0x82,0xc9,0x7d,0xfa,0x59,0x47,0xf0,0xad,0xd4,0xa2,0xaf,0x9c,0xa4,0x72,0xc0,
    0xb7,0xfd,0x93,0x26,0x36,0x3f,0xf7,0xcc,0x34,0xa5,0xe5,0xf1,0x71,0xd8,0x31,0x15,
    0x04,0xc7,0x23,0xc3,0x18,0x96,0x05,0x9a,0x07,0x12,0x80,0xe2,0xeb,0x27,0xb2,0x75,
    0x09,0x83,0x2c,0x1a,0x1b,0x6e,0x5a,0xa0,0x52,0x3b,0xd6,0xb3,0x29,0xe3,0x2f,0x84,
    0x53,0xd1,0x00,0xed,0x20,0xfc,0xb1,0x5b,0x6a,0xcb,0xbe,0x39,0x4a,0x4c,0x58,0xcf,
    0xd0,0xef,0xaa,0xfb,0x43,0x4d,0x33,0x85,0x45,0xf9,0x02,0x7f,0x50,0x3c,0x9f,0xa8,
    0x51,0xa3,0x40,0x8f,0x92,0x9d,0x38,0xf5,0xbc,0xb6,0xda,0x21,0x10,0xff,0xf3,0xd2,
    0xcd,0x0c,0x13,0xec,0x5f,0x97,0x44,0x17,0xc4,0xa7,0x7e,0x3d,0x64,0x5d,0x19,0x73,
    0x60,0x81,0x4f,0xdc,0x22,0x2a,0x90,0x88,0x46,0xee,0xb8,0x14,0xde,0x5e,0x0b,0xdb,
    0xe0,0x32,0x3a,0x0a,0x49,0x06,0x24,0x5c,0xc2,0xd3,0xac,0x62,0x91,0x95,0xe4,0x79,
    0xe7,0xc8,0x37,0x6d,0x8d,0xd5,0x4e,0xa9,0x6c,0x56,0xf4,0xea,0x65,0x7a,0xae,0x08,
    0xba,0x78,0x25,0x2e,0x1c,0xa6,0xb4,0xc6,0xe8,0xdd,0x74,0x1f,0x4b,0xbd,0x8b,0x8a,
    0x70,0x3e,0xb5,0x66,0x48,0x03,0xf6,0x0e,0x61,0x35,0x57,0xb9,0x86,0xc1,0x1d,0x9e,
    0xe1,0xf8,0x98,0x11,0x69,0xd9,0x8e,0x94,0x9b,0x1e,0x87,0xe9,0xce,0x55,0x28,0xdf,
    0x8c,0xa1,0x89,0x0d,0xbf,0xe6,0x42,0x68,0x41,0x99,0x2d,0x0f,0xb0,0x54,0xbb,0x16
], dtype=np.uint8)
INV_SBOX = np.empty_like(SBOX)
INV_SBOX[SBOX] = np.arange(256, dtype=np.uint8)

RCON = np.array([0x00,0x01,0x02,0x04,0x08,0x10,0x20,0x40,0x80,0x1B,0x36], dtype=np.uint8)

def rot_word(w: np.ndarray) -> np.ndarray:
    return np.roll(w, -1)

def sub_word(w: np.ndarray) -> np.ndarray:
    return SBOX[w]

def xtime(x: np.ndarray) -> np.ndarray:
    return ((x << 1) & 0xFE) ^ ((x >> 7) * 0x1B)

def gm_mul(a: np.ndarray, b: int) -> np.ndarray:
    # multiply vector of bytes a by constant b in GF(2^8)
    if b == 1: return a
    if b == 2: return xtime(a)
    if b == 3: return xtime(a) ^ a
    if b == 9: return xtime(xtime(xtime(a))) ^ a
    if b == 11: return xtime(xtime(xtime(a)) ^ a) ^ a
    if b == 13: return xtime(xtime(xtime(a) ^ a)) ^ a
    if b == 14: return xtime(xtime(xtime(a) ^ a) ^ a)
    raise ValueError("Unsupported mul")

# AES state is 16 bytes in column-major order (like the spec and your RTL):
# indices in a column: [0,4,8,12], [1,5,9,13], [2,6,10,14], [3,7,11,15]
COLS = [[0,4,8,12],[1,5,9,13],[2,6,10,14],[3,7,11,15]]

def shift_rows(state: np.ndarray) -> np.ndarray:
    out = state.copy()
    out[[1,5,9,13]] = state[[5,9,13,1]]
    out[[2,6,10,14]] = state[[10,14,2,6]]
    out[[3,7,11,15]] = state[[15,3,7,11]]
    return out

def inv_shift_rows(state: np.ndarray) -> np.ndarray:
    out = state.copy()
    out[[1,5,9,13]] = state[[13,1,5,9]]
    out[[2,6,10,14]] = state[[10,14,2,6]]  # 2-step rotation is symmetric
    out[[3,7,11,15]] = state[[7,11,15,3]]
    return out

def mix_columns(state: np.ndarray) -> np.ndarray:
    out = state.copy()
    for c in range(4):
        idx = COLS[c]
        s = state[idx]
        out[idx[0]] = (gm_mul(s,2) ^ gm_mul(s[[1]],3) ^ s[[2]] ^ s[[3]])[0]
        out[idx[1]] = (s[[0]] ^ gm_mul(s,2) ^ gm_mul(s[[2]],3) ^ s[[3]])[1-0]  # adjust indexing
        # cleaner version:
        out[idx[1]] = (s[0] ^ gm_mul(s[1:2],2)[0] ^ gm_mul(s[2:3],3)[0] ^ s[3]).astype(np.uint8)
        out[idx[2]] = (s[0] ^ s[1] ^ gm_mul(s[2:3],2)[0] ^ gm_mul(s[3:4],3)[0]).astype(np.uint8)
        out[idx[3]] = (gm_mul(s[0:1],3)[0] ^ s[1] ^ s[2] ^ gm_mul(s[3:4],2)[0]).astype(np.uint8)
    return out

def sub_bytes(state: np.ndarray) -> np.ndarray:
    return SBOX[state]

def add_round_key(state: np.ndarray, rk: np.ndarray) -> np.ndarray:
    return state ^ rk

# ---------- Key expansion (AES-128 → 11 round keys, 16 bytes each) ----------
def expand_key_128(key_bytes: bytes) -> np.ndarray:
    assert len(key_bytes) == 16
    w = np.frombuffer(key_bytes, dtype=np.uint8).copy()
    # 44 words (4-byte), but we’ll output 11 round keys (11*16 bytes)
    rk = np.empty((11,16), dtype=np.uint8)
    rk[0] = w
    temp = w.copy()
    for i in range(1,11):
        t = temp[-4:].copy()
        t = sub_word(rot_word(t))
        t[0] ^= RCON[i]
        block = np.empty(16, dtype=np.uint8)
        block[0:4]   = temp[0:4]   ^ t
        block[4:8]   = temp[4:8]   ^ block[0:4]
        block[8:12]  = temp[8:12]  ^ block[4:8]
        block[12:16] = temp[12:16] ^ block[8:12]
        rk[i] = block
        temp = block
    return rk  # rk[0]=rk0 (master key), rk[10]=rk10

# ---------- Forward simulate to round 6 and extract RTL-like nodes ----------
def intermediates_round6(plaintexts: List[bytes], key: bytes, rond = 6) -> Dict[str, np.ndarray]:
    """
    Returns dict with:
      sbb_o6  : N x 16  (SubBytes outputs at start of round 6)
      shr_o6  : N x 16
      mxc_o6  : N x 16
      state6  : N x 16  (end of round 6 = mxc_o6 XOR rk6)
      rk5, rk6: 16-byte arrays (for reference)
    """
    rk = expand_key_128(key)  # rk0..rk10
    rk5, rk6,rk7 = rk[rond-1], rk[rond],rk[rond+1]

    N = len(plaintexts)
    sbb_o6 = np.zeros((N,16), dtype=np.uint8)
    shr_o6 = np.zeros_like(sbb_o6)
    mxc_o6 = np.zeros_like(sbb_o6)
    state6 = np.zeros_like(sbb_o6)
    
    sbb_o7 = np.zeros((N,16), dtype=np.uint8)
    shr_o7 = np.zeros_like(sbb_o6)
    mxc_o7 = np.zeros_like(sbb_o6)
    state7 = np.zeros_like(sbb_o6)

    for i, P in enumerate(plaintexts):
        state = np.frombuffer(P, dtype=np.uint8).copy()

        # round 0: initial AddRoundKey
        state = add_round_key(state, rk[0])
        
        # rounds 1..5 (full rounds)
        for rnd in range(1,rond):
#             print(f"rnd= {rnd}")
            state = sub_bytes(state)
            state = shift_rows(state)
            state = mix_columns(state)
            state = add_round_key(state, rk[rnd])

        # start of round 6: SubBytes on state (this matches sbb_o when round==6)
        sb = sub_bytes(state)       # sbb_o6
        sbb_o6[i] = sb
        sh = shift_rows(sb)         # shr_o6
        shr_o6[i] = sh
        mx = mix_columns(sh)        # mxc_o6
        mxc_o6[i] = mx
        st6 = add_round_key(mx, rk6)  # state_new at end of round 6
        state6[i] = st6
        
                # start of round 6: SubBytes on state (this matches sbb_o when round==6)
        sb = sub_bytes(st6)       # sbb_o6
        sbb_o7[i] = sb
        sh = shift_rows(sb)         # shr_o6
        shr_o7[i] = sh
        mx = mix_columns(sh)        # mxc_o6
        mxc_o7[i] = mx
        st7 = add_round_key(mx, rk7)  # state_new at end of round 6
        state7[i] = st7

    return {
        "sbb_o6": sbb_o6,
        "shr_o6": shr_o6,
        "mxc_o6": mxc_o6,
        "state6": state6,
        "sbb_o7": sbb_o7,
        "shr_o7": shr_o7,
        "mxc_o7": mxc_o7,
        "state7": state7,
        "rk5": rk5.copy(),
        "rk7": rk7.copy(),
        "rk6": rk6.copy()
    }


# ---------- CPA helpers ----------
def hamming_weight(x: np.ndarray) -> np.ndarray:
    return np.unpackbits(x.reshape(-1,1), axis=1).sum(axis=1).astype(np.int16)

def column_indices(col: int) -> List[int]:
    return COLS[col]

def predict_round6_sbox_column_sumHW(plaintexts: List[bytes], guess_rk5_col: bytes, key: bytes, col: int) -> np.ndarray:
    """
    Prediction scalar per trace for CPA target aligned to sbb_o at round 6, one column at a time.
    We simulate rounds 0..5 using the guessed 4 bytes of rk5 for the selected column, and the true key for other columns.
    If you prefer pure-guessing without using the true key elsewhere, set 'key' to any value but also overwrite rk[5] col.
    """
    rk = expand_key_128(key)
    # overwrite the 4 bytes of rk5 for the chosen column with the guess
    idx = column_indices(col)
    rk[5] = rk[5].copy()
    rk[5][idx] = np.frombuffer(guess_rk5_col, dtype=np.uint8)

    preds = np.zeros(len(plaintexts), dtype=np.float64)
    for i, P in enumerate(plaintexts):
        state = np.frombuffer(P, dtype=np.uint8).copy()
        state = add_round_key(state, rk[0])
        for rnd in range(1,6):
            state = sub_bytes(state)
            state = shift_rows(state)
            state = mix_columns(state)
            state = add_round_key(state, rk[rnd])
        sb = sub_bytes(state)  # sbb_o at round 6
        # sum HW of the 4 bytes in this column
        preds[i] = hamming_weight(sb[idx]).sum()
    return preds

def pearson_corr(x: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """
    x: (N,) predictions
    Y: (N, T) traces (windowed)
    returns per-sample correlation (T,)
    """
    x = x.astype(np.float64)
    X = (x - x.mean())
    denom_x = np.sqrt((X*X).sum())
    Yc = Y - Y.mean(axis=0, keepdims=True)
    denom_y = np.sqrt((Yc*Yc).sum(axis=0))
    denom = denom_x * denom_y
    denom[denom == 0] = np.inf
    return (X[:,None] * Yc).sum(axis=0) / denom

# ---------- CPA drivers ----------
def run_cpa_round6_column(plaintexts: List[bytes], traces: np.ndarray, window: Tuple[int,int], key_for_others: bytes, col: int, guess_bytes: List[int]=(0,1,2,3)):
    """
    Column-wise CPA at round 6 using sbb_o6 (sum HW over 4 bytes in the column).
    Brute-forces the selected positions in the column (default: all 4 → 2^32; for faster testing, pass fewer).
    - plaintexts: list of 16-byte plaintexts
    - traces: (N, T) array
    - window: (start, end) sample indices for the round-6 S-box activity
    - key_for_others: a 16-byte key used to compute other columns' round keys (does not need to be true if you guess all 4 bytes)
    - col: which column [0..3]
    - guess_bytes: positions inside the column to brute-force (subset of [0,1,2,3])
    Returns: (best_guess_4bytes, best_corr_value, corr_waveform)
    """
    idx = column_indices(col)
    start, end = window
    Y = traces[:, start:end]

    # prepare a template for the 4-byte guess; unknowns looped, knowns taken from expanded rk5 of key_for_others
    rk = expand_key_128(key_for_others)
    base_col = rk[5][idx].copy()

    # build search space
    positions = list(guess_bytes)
    n_guess = 256 ** len(positions)
    best = (None, -np.inf, None)

    # simple nested loop via np.ndindex
    for vals in np.ndindex(*(256,)*len(positions)):
        guess_col = base_col.copy()
        for p, v in zip(positions, vals):
            guess_col[p] = v
        preds = predict_round6_sbox_column_sumHW(plaintexts, guess_col.tobytes(), key_for_others, col)
        corr = pearson_corr(preds, Y)
        peak = np.max(np.abs(corr))
        if peak > best[1]:
            best = (guess_col.copy(), float(peak), corr.copy())

    return bytes(best[0].tolist()), best[1], best[2]

# ---------- Last-round single-byte baseline (optional) ----------
def run_cpa_last_round_byte(ciphertexts: List[bytes], traces: np.ndarray, window: Tuple[int,int], byte_index_postSR: int, model: str="inv_sbox"):
    """
    Classic last-round CPA per byte.
    model = "inv_sbox" → preds = HW( InvSbox( C[j] ^ k ) )
          = "xor"      → preds = HW( C[j] ^ k )
    Returns: (best_key_byte, best_corr_value, corr_waveform)
    """
    start, end = window
    Y = traces[:, start:end]
    Cj = np.frombuffer(b''.join(ct[byte_index_postSR:byte_index_postSR+1] for ct in ciphertexts), dtype=np.uint8)

    best = (None, -np.inf, None)
    for k in range(256):
        z = Cj ^ k
        if model == "inv_sbox":
            preds = hamming_weight(INV_SBOX[z])
        elif model == "xor":
            preds = hamming_weight(z)
        else:
            raise ValueError("model must be 'inv_sbox' or 'xor'")
        corr = pearson_corr(preds.astype(np.float64), Y)
        peak = np.max(np.abs(corr))
        if peak > best[1]:
            best = (k, float(peak), corr.copy())
    return best
